# Capítulo 13: Bagging, Florestas e Boosting

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde à seção 8.2 de James et al. (2023).

> O elemento decisivo é a instabilidade do método de previsão. Se perturbar o conjunto de aprendizado pode causar mudanças significativas no preditor construído, então o bagging pode melhorar a acurácia.
>
> — Leo Breiman, em tradução livre (1994)

Uma árvore crescida até o fim muda de estrutura quando a amostra de treino muda, e a média de muitas previsões oscila menos que cada uma delas. Como só existe uma amostra, o bagging fabrica outras: sorteia, com reposição, amostras bootstrap do tamanho do treino, ajusta uma árvore sem poda em cada uma e prevê com a média das árvores, ou com o voto, em classificação. A seção 13.1 faz isso nos jogadores de `Hitters` e mede a variância da previsão ao longo de 100 conjuntos de treino: 0,21 numa árvore ajustada numa amostra bootstrap e 0,03 no bagging de 50 árvores. A queda não chega à divisão por $B$ que árvores independentes dariam, porque as árvores saem dos mesmos jogadores e se parecem. Cada amostra bootstrap deixa de fora cerca de um terço das observações, fração que tende a $1/e$, e a seção 13.2 usa essa sobra nos pacientes de `Heart`: prevendo cada paciente só com as árvores que não o viram, o erro *out-of-bag* estima o erro de teste com um único ajuste, onde a validação cruzada de 10 grupos pede dez.

No bagging dos jogadores, uma mesma coluna, `vezes_ao_bastao_carreira`, está na raiz de quase metade das árvores. A floresta aleatória, na seção 13.3, sorteia a cada divisão $m$ das $p$ colunas como candidatas ao corte, em geral $m \approx \sqrt{p}$; com $m = 4$ das 19 colunas, 15 colunas diferentes chegam à raiz de alguma árvore, contra 6 no bagging. Cada árvore fica um pouco pior e as árvores ficam menos parecidas, e a média saiu ganhando na maior parte das divisões medidas, nos jogadores e nos pacientes, sem ganhar em todas. O $m$ se escolhe pelo erro *out-of-bag*, com um ajuste por valor.

Um modelo de centenas de árvores não se lê como uma árvore, e a seção 13.5 pergunta em que colunas ele se apoia. A importância por redução de impureza soma o quanto os cortes de cada coluna separam as observações de treino; a importância por permutação mede quanto o acerto em dados novos cai quando a coluna é embaralhada. Nos pacientes, as duas põem o exame com tálio em primeiro e discordam sobre o colesterol, que acumula importância por impureza em cortes fundos, perto das folhas, e cujo embaralhamento não piora o acerto no teste. Nos jogadores, duas colunas de carreira quase iguais dividem a importância de um jeito no bagging e de outro na floresta.

O boosting não sorteia nada. Parte de uma previsão igual para todos e ajusta árvores pequenas em sequência, cada uma aos resíduos que as anteriores deixaram, somando ao modelo só uma fração $\lambda$ de cada árvore. A seção 13.4 percorre os três parâmetros nos jogadores: com $\lambda$ menor, o menor erro de teste chega mais tarde, em $B = 51$ árvores com $\lambda = 0{,}1$ e em $B = 635$ com $\lambda = 0{,}01$; passado o mínimo, o erro de teste volta a subir, e por isso o $B$ se escolhe por validação cruzada; e o número de divisões $d$ de cada árvore limita a ordem das interações que o modelo captura. A seção 13.6 põe os três métodos ao lado de uma árvore podada e de um modelo linear, na mesma validação cruzada de 10 grupos. Nos jogadores, os três ensembles erram menos que a árvore podada em todos os grupos; nos pacientes, a regressão logística fica com a menor taxa de erro, e o boosting perto dela. Nenhum método ganhou nos dois conjuntos, e o custo de escolher os parâmetros é diferente em cada um: no bagging basta um $B$ grande, na floresta o erro *out-of-bag* escolhe o $m$, e no boosting $B$, $\lambda$ e $d$ interagem e pedem validação cruzada.

Ao final deste capítulo, você será capaz de:

- Explicar por que a média de muitas previsões oscila menos que cada uma, sortear amostras bootstrap com `numpy` e ajustar o bagging de árvores com `BaggingRegressor` e com o `RandomForestClassifier` com `max_features=None`
- Estimar o erro de teste de um bagging pelo erro *out-of-bag* e compará-lo com a validação cruzada e com um conjunto de teste
- Ajustar uma floresta aleatória, explicar como o sorteio de $m$ colunas por divisão torna as árvores menos parecidas e escolher $m$ pelo erro *out-of-bag*
- Descrever o algoritmo do boosting, ajustar um `GradientBoostingRegressor`, interpretar o papel de $B$, $\lambda$ e $d$ e escolher $B$ por validação cruzada com `staged_predict`
- Calcular e interpretar a importância por redução de impureza e a importância por permutação (`permutation_importance`), reconhecendo o efeito dos cortes fundos e das colunas que se substituem
- Comparar bagging, floresta e boosting com uma árvore e um modelo linear por validação cruzada, e decidir o que precisa ser escolhido em cada método

## Seções

| Seção | Tópico |
|---|---|
| [13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-bagging.html) | Bagging |
| [13.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-erro-out-of-bag.html) | Erro Out-of-Bag |
| [13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-florestas-aleatorias.html) | Florestas Aleatórias |
| [13.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-boosting.html) | Boosting |
| [13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-importancia-de-variaveis.html) | Importância de Variáveis |
| [13.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/06-resumo-dos-metodos-de-ensemble.html) | Resumo dos Métodos de Ensemble |

## Bagging

> **📌 Nota**
>
> Esta seção corresponde à seção 8.2.1 de James et al. (2023).

Uma árvore de regressão crescida até o fim prevê o salário de um jogador de beisebol de um jeito com uma amostra de treino e de outro jeito com outra. Se houvesse muitas amostras de treino, daria para ajustar uma árvore em cada uma e tirar a média das previsões, e a média oscilaria menos que cada árvore. Mas só existe uma amostra. Dá para fabricar outras a partir dela?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.tree import DecisionTreeRegressor

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Média de muitas previsões

Tirar a média reduz a variância. Se uma medida tem variância $\sigma^2$, a média de 4 medidas independentes tem variância $\sigma^2/4$, e a de 100, $\sigma^2/100$. Com $n$ medidas independentes, a média tem variância $\sigma^2/n$. A mesma conta vale para previsões. Com $B$ conjuntos de treino separados, tirados da mesma população, daria para ajustar um modelo em cada um, obter as previsões $\hat f^1(x), \hat f^2(x), \dots, \hat f^B(x)$ e ficar com a média delas,

$$
\hat f_{\text{avg}}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat f^b(x),
$$

um modelo de variância menor que a de cada $\hat f^b$. A árvore de regressão sem limite de tamanho é a candidata natural: crescida até o fim, ela tem viés baixo e variância alta, porque muda de estrutura quando a amostra muda.

O obstáculo é que quase nunca há $B$ conjuntos de treino. Há um, e o que se pode fazer é sortear, a partir dele, outros conjuntos parecidos.

### Reamostrar com reposição

Os dados são os jogadores de `Hitters` com salário registrado. A resposta `y` é o logaritmo do salário, que está em milhares de dólares, e as colunas de texto viram indicadoras. O treino tem 132 jogadores sorteados com a semente 12, e o teste, os outros:

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
y = np.log(hitters["salario"])
X = pd.get_dummies(hitters.drop(columns="salario"),
                   drop_first=True, dtype=float)
treino_X, teste_X, treino_y, teste_y = train_test_split(
    X, y, train_size=132, random_state=12)
print("jogadores:", len(hitters), "  colunas:", X.shape[1])
print("treino:", len(treino_X), "  teste:", len(teste_X))

> **🔧 Função**
>
> **`pd.read_csv(caminho)`** — lê um arquivo CSV e devolve um `DataFrame`. **`df.dropna()`** — a tabela sem as linhas que têm algum valor ausente. **`np.log(x)`** — o logaritmo natural de cada elemento.
>
> **`df.drop(columns=coluna)`** — a tabela sem a coluna dada. **`pd.get_dummies(df, drop_first, dtype)`** — troca cada coluna de texto por colunas indicadoras de 0 e 1, uma por categoria; `drop_first=True` descarta a primeira categoria de cada coluna, que fica implícita nos zeros das outras, e `dtype=float` grava 0,0 e 1,0. **`X.shape`** — o par (linhas, colunas); `X.shape[1]` é o número de colunas.
>
> **`train_test_split(X, y, train_size, random_state)`** — sorteia quais linhas vão para o treino e quais para o teste; `train_size=132` põe 132 linhas no treino, e `random_state` fixa o sorteio.

Uma amostra nova, do mesmo tamanho do treino, sai de sortear 132 vezes um jogador entre os 132, **com reposição**: o jogador sorteado volta para a urna e pode sair de novo. É a ***amostra bootstrap***. Uns jogadores aparecem duas ou três vezes, outros não aparecem. Antes de rodar o chunk: se o sorteio é repetido 132 vezes entre 132 jogadores, quantos você espera que fiquem de fora?

In [ ]:
# Figura: Quantas vezes cada um dos 132 jogadores de treino aparece numa amostra bootstrap. A barra cinza são os jogadores que ficaram de fora; as azuis, os que entraram uma ou mais vezes. O número sobre cada barra é a contagem de jogadores.
rng = np.random.default_rng(13)
amostra = rng.choice(132, size=132, replace=True)
print("primeiras:", amostra[:5])
distintos = len(np.unique(amostra))
print("jogadores distintos:", distintos)
print("fração dentro:", numero_br(distintos / 132, 3))
contagem = np.bincount(np.bincount(amostra, minlength=132))
alturas = [*contagem[:4], contagem[4:].sum()]
print("0 a 4+ vezes:", [int(a) for a in alturas])
cores = ["0.45", "C0", "C0", "C0", "C0"]
fig, ax = plt.subplots(figsize=(3.8, 3.0))
barras = ax.bar(["0", "1", "2", "3", "4+"], alturas, color=cores, width=0.7)
for barra, altura in zip(barras, alturas):
    ax.text(barra.get_x() + barra.get_width() / 2, altura + 1, str(altura),
            ha="center", va="bottom")
ax.set_ylim(0, 60)
ax.set_xlabel("vezes que o jogador aparece")
ax.set_ylabel("jogadores")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.random.default_rng(semente)`** — um gerador de números aleatórios do `numpy`, com a semente que fixa o sorteio. **`rng.choice(n, size, replace)`** — sorteia `size` inteiros entre 0 e `n − 1`; com `replace=True`, o mesmo número pode sair mais de uma vez. Aqui, cada número é a posição de um jogador no treino.
>
> **`np.unique(x)`** — os valores distintos de `x`, em ordem crescente. **`np.bincount(x, minlength)`** — quantas vezes cada inteiro de 0 em diante aparece em `x`; `minlength=132` garante uma posição para cada jogador, mesmo para quem não saiu. Aplicado duas vezes, conta quantos jogadores apareceram 0, 1, 2… vezes. **`v.sum()`** — a soma dos valores de um array; `contagem[4:].sum()` junta os que apareceram 4 vezes ou mais. **`int(x)`** converte cada contagem num inteiro comum, para imprimir.
>
> **`plt.subplots(figsize=(largura, altura))`** — cria uma figura, com o tamanho em polegadas, e o eixo `ax` em que se desenha. **`ax.bar(rotulos, alturas, color, width)`** — uma barra por rótulo, com a altura e a cor dadas; devolve as barras. **`barra.get_x()`**, **`barra.get_width()`** — a borda esquerda e a largura de uma barra, para centralizar o texto sobre ela. **`ax.text(x, y, texto, ha, va)`** — escreve `texto` na posição `(x, y)`; `ha="center"` centraliza na horizontal, e `va="bottom"` põe o texto acima do ponto.
>
> **`ax.set_ylim`** — os limites do eixo vertical; **`ax.set_xlabel`**, **`ax.set_ylabel`** — os nomes dos eixos. **`plt.tight_layout()`** — ajusta as margens para que nada fique cortado. **`plt.show()`** — exibe a figura pronta.

Nesta amostra, 80 dos 132 jogadores aparecem ao menos uma vez, uma fração de 0,606. Os outros 132 − 80 = 52 ficaram de fora, uma fração de 1 − 0,606 = 0,394. Essa fração muda de uma amostra para outra, mas em torno de um valor fixo. Cada sorteio deixa de pegar um jogador fixo com probabilidade $1 - 1/n$, e os $n$ sorteios são independentes, então a probabilidade de ele não sair nenhuma vez é $(1 - 1/n)^n$. Com $n$ grande, ela se aproxima de $1/e$. O chunk calcula a fórmula para $n = 132$ e a compara com a média de mil amostras sorteadas pelo mesmo gerador:

In [ ]:
fora = (1 - 1 / 132) ** 132
print("fora, pela fórmula:", numero_br(fora, 4))
print("1/e:", numero_br(np.exp(-1), 4))
dentro = [len(np.unique(rng.choice(132, size=132))) / 132
          for _ in range(1000)]
print("dentro, mil amostras:", numero_br(np.mean(dentro), 4))

> **🔧 Função**
>
> **`np.exp(x)`** — a exponencial; `np.exp(-1)` é $1/e$. **`np.mean(lista)`** — a média dos valores.

A fórmula dá 0,3665 fora, e a média das mil amostras, 0,6328 dentro, isto é, 1 − 0,6328 = 0,3672 fora. Em média, um jogador fica fora de pouco mais de um terço das amostras bootstrap, e é essa sobra que permite estimar o erro de teste sem separar mais jogadores (o erro *out-of-bag*, isto é, medido nos jogadores que ficaram fora de cada amostra, na seção 13.2).

### Bagging de árvores

Com as amostras bootstrap no lugar dos $B$ conjuntos de treino que não existem, a receita é a mesma: ajustar o modelo na $b$-ésima amostra e tirar a média das previsões. O asterisco em $\hat f^{*b}$ marca que o modelo foi ajustado numa amostra bootstrap, e não num conjunto de treino separado como o $\hat f^b$ de antes.

> **🔷 Conceito**
>
> O ***bagging*** (de *bootstrap aggregation*, agregação de amostras bootstrap) sorteia $B$ amostras bootstrap do treino, ajusta um modelo em cada uma e prevê com a média
>
> $$
> \hat f_{\text{bag}}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat f^{*b}(x).
> $$
>
> Com árvores de regressão, cada árvore cresce sem limite e **não é podada**: tem viés baixo e variância alta, e a média das $B$ árvores ataca a variância.

Com uma resposta qualitativa, como doente ou não, a média de classes não se calcula. A forma mais simples de agregar é o **voto majoritário**: cada uma das $B$ árvores prevê uma classe, e a previsão do bagging é a classe mais frequente entre elas.

Para ter com o que comparar, o chunk abaixo ajusta no treino de 132 jogadores as duas árvores da seção 12.2: a árvore sem limite e a árvore podada por custo-complexidade, com o `ccp_alpha` escolhido por validação cruzada de 6 grupos:

In [ ]:
grande = DecisionTreeRegressor(random_state=12).fit(treino_X, treino_y)
alfas = grande.cost_complexity_pruning_path(
    treino_X, treino_y).ccp_alphas
grade = np.r_[0.0, np.sqrt(alfas[1:-1] * alfas[2:])][::-1]
busca = GridSearchCV(DecisionTreeRegressor(random_state=12),
                     {"ccp_alpha": grade},
                     cv=KFold(6, shuffle=True, random_state=12),
                     scoring="neg_mean_squared_error")
podada = busca.fit(treino_X, treino_y).best_estimator_
print("folhas, sem limite:", grande.get_n_leaves())
print("folhas, podada:", podada.get_n_leaves())

> **🔧 Função**
>
> **`DecisionTreeRegressor(random_state)`** — a árvore de regressão; sem `max_leaf_nodes` nem `ccp_alpha`, ela cresce até não ter mais o que dividir. `random_state` fixa a ordem em que as colunas são examinadas, que decide empates entre cortes igualmente bons. **`modelo.fit(X, y)`** — ajusta o modelo e devolve o próprio modelo. **`arvore.get_n_leaves()`** — o número de folhas.
>
> **`arvore.cost_complexity_pruning_path(X, y).ccp_alphas`** — os valores de `ccp_alpha` em que um galho da árvore cai; a grade toma um ponto entre cada dois consecutivos. **`np.r_[a, b]`** — junta valores e arrays num array só. **`np.sqrt(x)`** — a raiz quadrada de cada elemento.
>
> **`KFold(k, shuffle, random_state)`** — divide as linhas em `k` grupos sorteados. **`GridSearchCV(estimador, grade, cv, scoring)`** — o erro de validação cruzada do `estimador` em cada valor da `grade`; `.best_estimator_` é o modelo reajustado no treino inteiro com o melhor valor.

Agora o bagging de 50 árvores sem limite, no mesmo treino, e o erro de teste dos três modelos nos 131 jogadores de fora:

In [ ]:
bagging = BaggingRegressor(DecisionTreeRegressor(), n_estimators=50,
                           random_state=13).fit(treino_X, treino_y)
modelos = {"sem limite": grande, "podada": podada, "bagging": bagging}
erro = {nome: mean_squared_error(teste_y, m.predict(teste_X))
        for nome, m in modelos.items()}
for nome, valor in erro.items():
    print(f"{nome}: {numero_br(valor, 4)}")
print("bagging < podada < sem limite:",
      erro["bagging"] < erro["podada"] < erro["sem limite"])

> **🔧 Função**
>
> **`BaggingRegressor(estimator, n_estimators, random_state)`** — o bagging: sorteia `n_estimators` amostras bootstrap do treino, ajusta uma cópia de `estimator` em cada uma e prevê com a média das previsões. `random_state` fixa os sorteios.
>
> **`modelo.predict(X_novo)`** — as previsões para as linhas de `X_novo`. **`mean_squared_error(y, previsto)`** — o erro quadrático médio.

A árvore sem limite tem mais de uma centena de folhas; a podada tem 4 e erra 0,3630 no teste. O bagging de 50 árvores sem limite erra cerca de 0,31, menos que as duas nesta divisão. O ganho tem um preço: a média de 50 árvores não é uma árvore, e não há um diagrama para ler de cima a baixo, como o da seção 12.1. O que se recupera dele é quais colunas mais pesam nas divisões, a importância de variáveis da seção 13.5.

### Quantas árvores

O número de árvores $B$ é o único parâmetro novo. O chunk ajusta o bagging no mesmo treino com $B$ de 1 a 500, todos com a mesma semente, e mede o erro nos 131 jogadores do teste:

In [ ]:
tamanhos = [1, 5, 10, 25, 50, 100, 200, 300, 500]
erro_B = pd.Series({
    B: mean_squared_error(teste_y, BaggingRegressor(
        DecisionTreeRegressor(), n_estimators=B, random_state=13)
        .fit(treino_X, treino_y).predict(teste_X))
    for B in tamanhos})
for B, valor in erro_B.items():
    print(f"B = {B}: {numero_br(valor, 4)}")
print("B = 1 acima da sem limite:",
      erro_B.loc[1] > erro["sem limite"])
print("B ≥ 5 abaixo da podada:",
      bool((erro_B.loc[5:] < erro["podada"]).all()))

> **🔧 Função**
>
> **`pd.Series({rotulo: valor, ...})`** — uma série a partir de um dicionário; aqui, o erro de teste com $B$ como rótulo. **`s.items()`** — percorre os pares (rótulo, valor). **`s.loc[5:]`** — as linhas do rótulo 5 em diante. **`(s < v).all()`** — `True` se a comparação vale em todas as linhas; `bool` o imprime como `True` ou `False`.

Com $B = 1$, o bagging é uma única árvore sem limite ajustada numa amostra bootstrap, e o erro de teste é 0,4080, acima do da árvore sem limite ajustada nos 132 jogadores. A amostra bootstrap tem só parte dos jogadores distintos, 80 dos 132 na amostra sorteada acima, e uma árvore que vê menos jogadores erra mais. Com 5 árvores o erro já cai para cerca de 0,32, e de 5 árvores em diante todo bagging da grade fica abaixo da árvore podada. A figura mostra a curva inteira:

In [ ]:
# Figura: O MSE de teste do bagging contra o número de árvores $B$, em escala logarítmica (azul). As linhas cinza são a árvore podada por validação cruzada (tracejada) e a árvore sem limite (pontilhada), as duas ajustadas no mesmo treino de 132 jogadores.
fig, ax = plt.subplots(figsize=(3.8, 3.2))
ax.plot(erro_B.index, erro_B.values, color="C0", marker="o")
ax.axhline(erro["podada"], color="0.45", linestyle="--", linewidth=1.5)
ax.axhline(erro["sem limite"], color="0.45", linestyle=":", linewidth=1.5)
ax.text(600, erro_B.iloc[-1], "bagging", va="center")
ax.text(600, erro["podada"] - 0.002, "podada", va="top")
ax.text(600, erro["sem limite"] + 0.002, "sem\nlimite", va="bottom")
ax.set_xscale("log")
ax.set_xlim(0.8, 3000)
ax.set_xticks([1, 10, 100, 500], ["1", "10", "100", "500"])
ax.set_ylim(0.25, 0.45)
ax.set_yticks(np.arange(0.25, 0.46, 0.05))
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.2f}".replace(".", ","))
ax.set_xlabel("número de árvores B")
ax.set_ylabel("MSE de teste")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.plot(x, y, color, marker)`** — uma linha ligando os pontos, com um marcador em cada um. **`s.index`**, **`s.values`** — os rótulos e os valores de uma série. **`s.iloc[-1]`** — o último valor da série, pela posição.
>
> **`ax.axhline(y, color, linestyle)`** — uma linha horizontal na altura `y`; `"--"` a faz tracejada, `":"`, pontilhada. **`ax.set_xscale("log")`** — eixo horizontal em escala logarítmica. **`ax.set_xlim`** — os limites do eixo horizontal. **`ax.set_xticks(posicoes, rotulos)`**, **`ax.set_yticks(posicoes)`** — as marcas de cada eixo e, no horizontal, o texto de cada uma. **`np.arange(inicio, fim, passo)`** — os valores de `inicio` até antes de `fim`, de `passo` em `passo`. **`ax.yaxis.set_major_formatter(funcao)`** — escreve cada marca do eixo vertical com a função dada, que recebe o valor e a posição; aqui, com vírgula decimal.

A curva é uma realização só: um treino, uma semente e os 131 jogadores do teste, e outra divisão daria outros números. A maior parte da queda acontece até 10 árvores, de 0,4080 com uma a cerca de 0,31 com dez. De 10 a 50 árvores o erro fica em torno de 0,31, e com 500 árvores fica abaixo de 0,30. Nos pontos medidos, aumentar $B$ não fez o erro voltar ao nível da árvore podada. Nesta amostra, portanto, $B$ não é um parâmetro delicado: basta que seja grande o bastante para o erro parar de cair depressa. O que impede de usar um $B$ enorme é o custo: cada árvore a mais é mais um ajuste, e o tempo cresce com $B$.

### Por que a média funciona

O bagging erra menos que a árvore porque as previsões dele oscilam menos. A **variância de uma previsão** é o quanto $\hat f(x_0)$ muda de um conjunto de treino para outro, num jogador $x_0$ que o modelo não viu.

Para medi-la, o chunk abaixo sorteia 100 conjuntos de treino de 132 jogadores entre os 263 e ajusta, em cada um, três modelos: a árvore sem limite, crescida no treino inteiro; uma árvore bootstrap, isto é, o bagging com $B = 1$, feita do mesmo jeito que cada uma das árvores que o bagging junta; e o bagging de 50 árvores. Cada modelo prevê os 131 jogadores que ficaram fora daquele treino.

No fim, cada jogador tem as previsões dos conjuntos em que não entrou. A variância delas é a variância da previsão para ele, e a média do quadrado da diferença entre cada previsão e o log do salário observado é o erro médio dele fora da amostra. Os 263 jogadores fazem o papel da população, e os 100 conjuntos se sobrepõem; por isso os números servem para comparar os modelos, não como a variância exata de cada um.

In [ ]:
posicoes = np.arange(len(X))
tres_modelos = {
    "sem limite": DecisionTreeRegressor(random_state=12),
    "bootstrap": BaggingRegressor(DecisionTreeRegressor(),
                                  n_estimators=1, random_state=13),
    "bagging": BaggingRegressor(DecisionTreeRegressor(),
                                n_estimators=50, random_state=13)}
previsoes = {nome: np.full((100, len(X)), np.nan)
             for nome in tres_modelos}
for s in range(100):
    dentro, fora = train_test_split(posicoes, train_size=132,
                                    random_state=s)
    for nome, modelo in tres_modelos.items():
        modelo.fit(X.iloc[dentro], y.iloc[dentro])
        previsoes[nome][s, fora] = modelo.predict(X.iloc[fora])
vezes_fora = (~np.isnan(previsoes["bagging"])).sum(axis=0)
print("previsões por jogador:", vezes_fora.min(), "a", vezes_fora.max())
var = {nome: np.nanvar(p, axis=0) for nome, p in previsoes.items()}
for nome, p in previsoes.items():
    erro_medio = np.nanmean((p - y.values) ** 2, axis=0)
    resto = (np.nanmean(p, axis=0) - y.values) ** 2
    print(f"{nome}: erro {numero_br(erro_medio.mean())}")
    print(f"  variância {numero_br(var[nome].mean())}"
          f" + resto {numero_br(resto.mean())}")
print("bagging < bootstrap:",
      (var["bagging"] < var["bootstrap"]).sum(), "de", len(X))

> **🔧 Função**
>
> **`np.full((linhas, colunas), np.nan)`** — uma matriz preenchida com `nan`, o valor ausente; cada linha é um conjunto de treino e cada coluna, um jogador, e só as posições de quem ficou fora recebem previsão. Passadas ao `train_test_split`, as posições de `posicoes` são divididas em `dentro` e `fora`.
>
> **`np.isnan(m)`** — `True` onde `m` tem `nan`; o `~` inverte, e `.sum(axis=0)` conta, em cada coluna, as previsões que existem. **`np.nanvar(m, axis=0)`**, **`np.nanmean(m, axis=0)`** — a variância e a média de cada coluna, deixando os `nan` de fora. **`y.values`** — os valores da série como array, para subtrair de cada linha da matriz. **`v.mean()`**, **`v.min()`**, **`v.max()`** — a média, o mínimo e o máximo de um array.

Cada jogador ficou fora de 37 a 63 dos 100 conjuntos de treino, e é sobre essas previsões que o chunk calcula, jogador a jogador, três números: o erro médio, a variância das previsões e o **resto**, o quadrado da distância entre o centro delas e o valor observado. O erro médio de cada jogador é exatamente a variância mais o resto, e as linhas impressas são as médias sobre os 263 jogadores, com duas casas; a soma impressa pode diferir do erro na última casa, pelo arredondamento.

A árvore bootstrap tem variância média de 0,21; o bagging de 50 árvores, de 0,03, e a variância do bagging fica abaixo da variância da árvore bootstrap em 262 dos 263 jogadores. A árvore bootstrap erra em média 0,41, mais que os 0,37 da árvore sem limite crescida no treino inteiro. Com $B = 1$, na divisão de treino e teste lá de cima, ela também tinha errado mais que a árvore sem limite, e isso não foi acaso daquela divisão.

A figura mostra o que a média resume em três jogadores, escolhidos pelo salário: o de menor salário, um de salário mediano e o de maior salário.

In [ ]:
# Figura: As previsões do logaritmo do salário (em milhares de dólares) de três jogadores, uma por conjunto de treino em que o jogador ficou de fora: a árvore bootstrap em cinza, o bagging de 50 árvores em azul. O traço laranja tracejado não é um modelo: é o log do salário observado do jogador. Os jogadores são o de menor salário, um de salário mediano e o de maior salário.
ordem = np.argsort(y.values, kind="stable")
tres = ordem[[0, len(ordem) // 2, -1]]
jitter = np.random.default_rng(13)
fig, ax = plt.subplots(figsize=(3.8, 3.4))
for i, jogador in enumerate(tres):
    for desvio, nome, cor in [(-0.18, "bootstrap", "0.45"),
                              (0.18, "bagging", "C0")]:
        pontos = previsoes[nome][:, jogador]
        pontos = pontos[~np.isnan(pontos)]
        x = i + desvio + jitter.uniform(-0.06, 0.06, len(pontos))
        ax.scatter(x, pontos, s=10, alpha=0.5, color=cor,
                   label=nome if i == 0 else "_nolegend_")
    ax.hlines(y.iloc[jogador], i - 0.38, i + 0.38, color="C1",
              linestyle="--", linewidth=2,
              label="observado" if i == 0 else "_nolegend_")
ax.set_xticks(range(3), ["menor", "mediano", "maior"])
ax.set_xlabel("salário do jogador")
ax.set_ylabel("log do salário")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1f}".replace(".", ","))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3,
          markerscale=2)
plt.tight_layout()
plt.show()
quinto, perto = [], []
for rotulo, jogador in zip(["menor", "mediano", "maior"], tres):
    v = [var[nome][jogador] for nome in ["bootstrap", "bagging"]]
    centro = [np.nanmean(previsoes[nome][:, jogador])
              for nome in ["bootstrap", "bagging"]]
    print(f"{rotulo}: variância {numero_br(v[0], 3)}"
          f" e {numero_br(v[1], 3)}")
    print(f"  centro {numero_br(centro[0])} e {numero_br(centro[1])};"
          f" observado {numero_br(y.iloc[jogador])}")
    quinto.append(v[1] < v[0] / 5)
    perto.append(abs(centro[0] - centro[1]) < 0.1)
print("bagging abaixo de 1/5:", all(quinto))
print("centros a menos de 0,1:", all(perto))

> **🔧 Função**
>
> **`np.argsort(v, kind)`** — as posições que põem `v` em ordem crescente; a primeira é a do menor valor, a do meio a de um valor mediano, e a última, `-1`, a do maior. Dois jogadores podem ter o mesmo salário, e `kind="stable"` mantém, entre valores iguais, a ordem original das linhas. **`abs(x)`** e **`all(lista)`** — o valor absoluto, e `True` se todos os itens da lista forem verdadeiros. **`rng.uniform(a, b, n)`** — `n` números sorteados entre `a` e `b`; aqui, um deslocamento horizontal pequeno para os pontos não caírem um em cima do outro.
>
> **`ax.scatter(x, y, s, alpha, color, label)`** — um ponto por par `(x, y)`, com tamanho `s` e opacidade `alpha`; `label` é o nome na legenda, e `"_nolegend_"` o deixa de fora. **`ax.hlines(y, xmin, xmax, linestyle)`** — um traço horizontal na altura `y`, de `xmin` a `xmax`.
>
> **`ax.legend(loc, bbox_to_anchor, ncol, markerscale)`** — a legenda, com o ponto `loc` dela preso a `bbox_to_anchor`, em frações do painel; `(0.5, 1.0)` a põe acima do gráfico, `ncol=3` enfileira os itens numa linha, e `markerscale=2` desenha os marcadores da legenda com o dobro do tamanho.

Cada par de linhas impressas traz a variância da árvore bootstrap e a do bagging, e depois o centro de cada nuvem de pontos, a média das previsões, e o log do salário observado. Uma árvore bootstrap sozinha prevê valores espalhados por uma faixa larga, porque o jogador cai numa folha diferente a cada conjunto de treino, e cada folha tem a sua média. O bagging tira a média de 50 dessas folhas, e nos três jogadores a variância dele fica abaixo de um quinto da variância da árvore bootstrap.

Nos três, as duas nuvens têm o centro a menos de 0,1 uma da outra. É o que se espera do método em qualquer jogador. Cada árvore do bagging é uma árvore bootstrap, e a média de 50 delas tem o mesmo valor esperado que uma só: o bagging reduz a oscilação das previsões em torno do centro, e o centro fica onde estava. A distância entre o centro e o traço laranja fica fora do alcance da média. No jogador de menor salário, os centros ficam acima do valor observado; no de maior salário, abaixo, porque as folhas misturam cada um com jogadores de salário mais próximo do meio. Parte da distância vem do próprio salário, que inclui a parte que nenhuma coluna explica, o erro irredutível $\varepsilon$ da seção 7.2.

A decomposição da seção 7.6 vale para o erro de teste **esperado**, a média sobre muitos conjuntos de treino, e o separa em variância, viés² e $\mathrm{Var}(\varepsilon)$; a variância medida aqui é o primeiro termo. O resto faz o papel dos outros dois juntos e quase não muda: 0,20 na árvore bootstrap e 0,19 no bagging. O erro médio cai de 0,41 a 0,22, uma queda de 0,41 − 0,22 = 0,19, e a variância cai 0,21 − 0,03 = 0,18: quase toda a queda do erro médio é queda de variância. A curva do erro contra $B$ não se reparte desse jeito, porque ela é um treino só; a decomposição fala da média sobre os treinos, e é essa média que os 100 conjuntos aproximam.

Se as 50 árvores fossem independentes, a variância do bagging cairia a $\sigma^2/B$, com $\sigma^2$ a variância de uma árvore bootstrap. O chunk faz a conta:

In [ ]:
print("uma árvore / 50:", numero_br(var["bootstrap"].mean() / 50, 4))
print("50 árvores:", numero_br(var["bagging"].mean(), 4))

A conta dá 0,0042; medido, dá 0,0271. As árvores saem todas dos mesmos 132 jogadores, e as amostras bootstrap repetem boa parte deles: as árvores se parecem, e a média de coisas parecidas reduz menos a variância que a média de coisas independentes. Seja $\rho$ a correlação, medida sobre muitos conjuntos de treino, entre as previsões de duas árvores do mesmo bagging para um mesmo jogador. Se todas as árvores têm a mesma variância $\sigma^2$ e cada par tem a mesma correlação $\rho$, a variância da média de $B$ árvores é

$$
\rho\,\sigma^2 + \frac{(1 - \rho)\,\sigma^2}{B}.
$$

Com $\rho = 0$ volta a $\sigma^2/B$. Com árvores parecidas, $\rho$ é positivo, e só a segunda parcela cai com $B$; a primeira, $\rho\sigma^2$, fica, por maior que seja $B$. A distância entre os 0,0271 medidos e os 0,0042 da conta com árvores independentes vem de $\rho$ ser positivo. Por isso, depois de algumas dezenas de árvores, somar árvores muda pouco: na curva do erro contra $B$, de 10 para 500 árvores, o erro de teste foi de cerca de 0,31 para menos de 0,30.

## Erro Out-of-Bag

> **📌 Nota**
>
> Esta seção corresponde à seção 8.2.1 de James et al. (2023).

Cada árvore do bagging sai de uma amostra bootstrap e não vê pouco mais de um terço das observações de treino. Para uma observação que uma árvore não viu, a previsão dessa árvore vale tanto quanto a previsão para uma observação nova. Juntando, para cada observação, só as previsões das árvores que não a viram, dá para estimar o erro de teste sem separar um conjunto de validação e sem ajustar o modelo dez vezes?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Os pacientes

O conjunto `Heart` tem medidas de pacientes que chegaram a um hospital com dor no peito, como a idade, o colesterol e o resultado de um exame com tálio, e a resposta `doenca_cardiaca`, "sim" ou "não", dada pela angiografia. As duas colunas de texto, `dor_no_peito` e `talio`, viram indicadoras de **todas** as categorias, porque cada indicadora é uma pergunta que a árvore pode fazer. Metade dos pacientes fica para o teste, sorteada com a semente 13 e com a mesma proporção de doentes nas duas metades:

In [ ]:
coracao = pd.read_csv("dados/Heart.csv").dropna()
y_c = coracao["doenca_cardiaca"]
X_c = pd.get_dummies(coracao.drop(columns="doenca_cardiaca"),
                     dtype=float)
treino_Xc, teste_Xc, treino_yc, teste_yc = train_test_split(
    X_c, y_c, test_size=0.5, random_state=13, stratify=y_c)
print("pacientes:", len(coracao))
print("medidas:", coracao.shape[1] - 1, "  colunas:", X_c.shape[1])
print("treino:", len(treino_Xc), "  teste:", len(teste_Xc))
print("erro do 'sempre não':",
      numero_br((teste_yc == "sim").mean(), 4))

> **🔧 Função**
>
> **`train_test_split(X, y, test_size, random_state, stratify)`** — com `test_size=0.5`, metade das linhas vai para o teste; `stratify=y_c` sorteia de modo que a proporção de cada classe de `y_c` seja quase a mesma no treino e no teste. **`serie == "sim"`** — uma série de `True` e `False`; a média dela é a fração de `True`.

Dos 297 pacientes com todas as medidas, 148 ficam no treino e 149 no teste. As 13 medidas viram 18 colunas depois das indicadoras. Um classificador que respondesse "não" para todo paciente, sem olhar medida nenhuma, erraria no teste uma fração de 0,4631. Um erro de classificação a mais ou a menos no teste muda a taxa em 1/149, e no treino, em 1/148.

### As árvores que não viram cada paciente

O bagging de árvores de classificação é o `RandomForestClassifier` com `max_features=None`: toda divisão de toda árvore pode usar qualquer uma das 18 colunas. Cada árvore sai de uma amostra bootstrap própria, e o chunk ajusta 300 delas e marca, a partir dos pacientes que cada amostra sorteou, quais ficaram de fora de cada árvore:

In [ ]:
bagging = RandomForestClassifier(n_estimators=300, max_features=None,
                                 oob_score=True, random_state=13)
bagging.fit(treino_Xc, treino_yc)
fora = np.ones((300, len(treino_Xc)), dtype=bool)
for b, sorteados in enumerate(bagging.estimators_samples_):
    fora[b, sorteados] = False
print("fração fora, média:", numero_br(fora.mean(), 4))
por_paciente = fora.sum(axis=0)
print("árvores que não viram o paciente:")
print("  mínimo", por_paciente.min(),
      " mediana", int(np.median(por_paciente)),
      " máximo", por_paciente.max())
print("todos têm árvores:", bool((por_paciente > 0).all()))
print("(1 − 1/148)^148:", numero_br((1 - 1 / 148) ** 148, 4))
print("1/e:", numero_br(np.exp(-1), 4))

> **🔧 Função**
>
> **`RandomForestClassifier(n_estimators, max_features, oob_score, random_state)`** — ajusta `n_estimators` árvores de classificação sem limite de profundidade, cada uma numa amostra bootstrap do treino. `max_features=None` deixa toda divisão escolher entre todas as colunas, e aí o modelo é o bagging; o padrão, `"sqrt"`, dá a floresta aleatória da seção 13.3. `oob_score=True` calcula, no próprio ajuste, o acerto *out-of-bag* definido logo abaixo. `random_state` fixa os sorteios.
>
> **`modelo.estimators_samples_`** — para cada árvore, as posições das linhas de treino que a amostra bootstrap sorteou, com repetição. **`np.ones((linhas, colunas), dtype=bool)`** — uma matriz toda `True`; aqui, uma linha por árvore e uma coluna por paciente, e as posições sorteadas viram `False`. **`np.median(v)`** — a mediana.

Em média, cada árvore deixou de fora uma fração de 0,3676 dos 148 pacientes. É o que o sorteio prevê: um paciente fica fora de uma amostra quando nenhum dos 148 sorteios o escolhe, o que acontece com chance $(1 - 1/148)^{148} = 0{,}3666$, e essa chance tende a $1/e = 0{,}3679$ quando o número de pacientes cresce. É pouco mais de um terço. Visto do lado do paciente, a conta é a mesma: das 300 árvores, cerca de 300 × 0,3666 = 110 não o viram. A mediana é 110, e mesmo o paciente com menos árvores tem 92. Todo paciente tem árvores que não o viram.

A matriz `fora` tem 300 linhas e 148 colunas. Um canto dela, as 20 primeiras árvores e os 10 primeiros pacientes do treino, mostra como as árvores que não viram cada paciente se espalham:

In [ ]:
# Figura: As 20 primeiras árvores do bagging (linhas) e os 10 primeiros pacientes do treino (colunas). Um quadrado laranja marca um paciente que ficou fora da amostra bootstrap daquela árvore; um ponto cinza, um paciente que a amostra sorteou uma ou mais vezes.
canto = fora[:20, :10]
arvores, pacientes = np.nonzero(canto)
dentro_a, dentro_p = np.nonzero(~canto)
fig, ax = plt.subplots(figsize=(3.6, 4.4))
ax.scatter(pacientes + 1, arvores + 1, marker="s", s=70, color="C1",
           label="fora da amostra (OOB)")
ax.scatter(dentro_p + 1, dentro_a + 1, s=8, color="0.45",
           label="sorteado")
ax.set_xlim(0.3, 10.7)
ax.set_ylim(20.7, 0.3)
ax.set_xticks(range(1, 11))
ax.set_yticks([1, 5, 10, 15, 20])
ax.grid(False)
ax.set_xlabel("paciente de treino")
ax.set_ylabel("árvore")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=1)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.nonzero(m)`** — as posições dos `True` de uma matriz, como dois arrays: as linhas e as colunas. **`~m`** — troca `True` por `False` e vice-versa. **`marker="s"`**, no `ax.scatter` — cada ponto é um quadrado. **`ax.grid(False)`** — tira a malha do fundo.

> **🔷 Conceito**
>
> Os pacientes que ficaram fora da amostra bootstrap de uma árvore são as observações ***out-of-bag*** (OOB, "fora da sacola") daquela árvore. A **previsão OOB** do paciente $i$ junta só as árvores em que $i$ foi OOB, cerca de $B/3$ delas (a fração tende a $1/e \approx 0{,}37$): a média das previsões, em regressão, ou o voto majoritário, em classificação. O **erro OOB** compara a previsão OOB de cada paciente de treino com a resposta dele: o MSE, em regressão, ou a taxa de erro de classificação. Nenhuma das árvores usadas para prever $i$ viu $i$, e por isso o erro OOB estima o erro de teste do bagging.

O `scikit-learn` decide pela média das probabilidades que as árvores dão a cada classe e fica com a classe de maior média. Uma árvore crescida até o fim tem folhas puras, só de doentes ou só de não doentes, e dá probabilidade 0 ou 1 a cada classe, isto é, um voto; a média das probabilidades de "sim" é a fração de árvores que votam "sim". Para o primeiro paciente do treino, a primeira coluna da figura, o ajuste guardou essa fração em `oob_decision_function_`, calculada só com as árvores que não o viram:

In [ ]:
oob = bagging.oob_decision_function_
print("classes:", list(bagging.classes_))
print("árvores que não o viram:", por_paciente[0])
print("fração que vota 'sim':", numero_br(oob[0, 1], 2))
print("votos 'sim':", round(oob[0, 1] * por_paciente[0]))
print("previsão OOB:", bagging.classes_[oob[0].argmax()])
print("resposta:", treino_yc.iloc[0])

> **🔧 Função**
>
> **`modelo.oob_decision_function_`** — a média OOB das probabilidades de cada classe, um paciente de treino por linha e uma coluna por classe, na ordem de `modelo.classes_` ("não", "sim"); `[0, 1]` é a fração de "sim" do primeiro paciente. **`v.argmax()`** — a posição do maior valor; num empate, a primeira.

Das 100 árvores que não viram o primeiro paciente, 96 votam "sim", e a previsão OOB dele é "sim", que é também a resposta da angiografia. Esse paciente conta como acerto no erro OOB. Se o número de árvores que não viram um paciente for par, ele pode ter tantos votos "sim" quanto "não"; aí o `argmax` fica com a primeira classe, "não". O erro OOB faz essa conta para os 148 pacientes:

In [ ]:
erro_oob = 1 - bagging.oob_score_
print("erro OOB:", numero_br(erro_oob, 4))
print("entre 0,21 e 0,24:", 0.21 <= erro_oob <= 0.24)

> **🔧 Função**
>
> **`modelo.oob_score_`** — o acerto OOB de um classificador, a fração de pacientes de treino cuja previsão OOB acerta; o erro é `1 − oob_score_`. Num regressor, como o `RandomForestRegressor`, o `oob_score_` é o $R^2$, e o MSE OOB se calcula com as previsões guardadas em `oob_prediction_`.

O erro OOB fica entre 0,21 e 0,24, bem abaixo do 0,4631 do "sempre não". O valor impresso pode mudar de um computador para outro por um paciente, 1/148. Árvores sem limite de profundidade encontram cortes igualmente bons, e quem desempata é a ordem em que as colunas são examinadas, sorteada a cada divisão; uma diferença de arredondamento muda um corte e, às vezes, a previsão OOB de um paciente.

### OOB, validação cruzada e teste

A validação cruzada de 10 grupos, a da seção 10.3, precisa de dez ajustes de 300 árvores, um por grupo. O conjunto de teste precisa de 149 pacientes que o modelo nunca usa. O chunk calcula as duas e as compara com o OOB. Antes de rodar: o OOB deve sair acima ou abaixo do teste?

In [ ]:
cv = cross_val_score(
    RandomForestClassifier(n_estimators=300, max_features=None,
                           random_state=13),
    treino_Xc, treino_yc,
    cv=StratifiedKFold(10, shuffle=True, random_state=13))
erro_cv = 1 - cv.mean()
erro_teste = 1 - bagging.score(teste_Xc, teste_yc)
print("CV de 10 grupos:", numero_br(erro_cv, 4))
print("teste:", numero_br(erro_teste, 4))
print("CV < OOB < teste:", erro_cv < erro_oob < erro_teste)

> **🔧 Função**
>
> **`StratifiedKFold(k, shuffle, random_state)`** — divide as linhas em `k` grupos sorteados, com a proporção de cada classe quase igual em todos. **`cross_val_score(modelo, X, y, cv)`** — ajusta o modelo `k` vezes, cada vez sem um grupo, e devolve o acerto em cada grupo deixado de fora; `cv.mean()` é o acerto médio. **`modelo.score(X, y)`** — num classificador, a fração de acertos em `(X, y)`.

A CV dá 0,2086, e o teste, 0,2483; o OOB fica entre as duas. As três são estimativas do mesmo erro, e nenhuma é o erro exato: a CV e o OOB usam os 148 pacientes de treino, o teste usa outros 149, e em cada uma um paciente vale cerca de 1/149 = 0,0067.

O que muda é o custo. O OOB saiu de um ajuste de 300 árvores; a CV precisou de dez, 3.000 árvores. Há também um resultado teórico: com $B$ grande, o erro OOB fica praticamente igual ao erro da validação cruzada *leave-one-out*, a da seção 10.2, que exigiria um ajuste por paciente (James et al., 2023). Num conjunto grande, em que dez ajustes do bagging já pesam, o OOB é a estimativa que sai sem ajuste extra.

O `oob_score_` depende do número de árvores. Com poucas, algum paciente pode ter ficado dentro de todas as amostras e não ter previsão OOB; por isso a curva começa em 20. O chunk abaixo faz crescer a mesma floresta de 10 em 10 árvores, de 20 até 300, e mede o erro OOB e o de teste em cada tamanho:

In [ ]:
crescendo = RandomForestClassifier(n_estimators=20, max_features=None,
                                   oob_score=True, warm_start=True,
                                   random_state=13)
tamanhos = range(20, 301, 10)
curva = pd.DataFrame(index=tamanhos, columns=["oob", "teste"],
                     dtype=float)
for B in tamanhos:
    crescendo.set_params(n_estimators=B).fit(treino_Xc, treino_yc)
    curva.loc[B] = [1 - crescendo.oob_score_,
                    1 - crescendo.score(teste_Xc, teste_yc)]
print("mesmo modelo do zero:", bool(
    (crescendo.predict_proba(teste_Xc)
     == bagging.predict_proba(teste_Xc)).all()))
print("teste, de", numero_br(curva["teste"].min(), 4),
      "a", numero_br(curva["teste"].max(), 4))
depois = curva.loc[100:]
print("de B = 100 em diante:")
print("  teste, de", numero_br(depois["teste"].min(), 4),
      "a", numero_br(depois["teste"].max(), 4))

> **🔧 Função**
>
> **`warm_start=True`** — num novo `fit`, mantém as árvores já ajustadas e só acrescenta as que faltam para chegar a `n_estimators`. Com a mesma semente, as árvores novas são as mesmas que um ajuste do zero produziria, e a floresta de 300 árvores sai igual à do chunk `fora-da-amostra`: é o que o booleano confere, nas probabilidades de teste. **`modelo.set_params(n_estimators=B)`** — muda um parâmetro do modelo antes do próximo `fit`.
>
> **`pd.DataFrame(index, columns, dtype)`** — uma tabela vazia, com os rótulos das linhas e das colunas; `curva.loc[B] = [...]` preenche a linha `B`, e `curva.loc[100:]` pega as linhas de rótulo 100 em diante. **`modelo.predict_proba(X)`** — a probabilidade de cada classe para cada linha.

In [ ]:
# Figura: Taxa de erro do bagging nos pacientes do `Heart` contra o número de árvores $B$: o erro OOB, calculado nos 148 pacientes de treino (azul tracejado), e o erro nos 149 pacientes de teste (laranja). O losango verde em $B = 300$ é o erro da validação cruzada de 10 grupos no treino.
fig, ax = plt.subplots(figsize=(3.8, 3.6))
ax.plot(curva.index, curva["oob"], color="C0", linestyle="--",
        label="OOB")
ax.plot(curva.index, curva["teste"], color="C1", label="teste")
ax.plot(300, erro_cv, "D", color="C2", markersize=7,
        label="CV, 10 grupos")
ax.set_xlim(0, 320)
ax.set_ylim(0.15, 0.30)
ax.set_yticks(np.arange(0.15, 0.31, 0.05))
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.2f}".replace(".", ","))
ax.set_xlabel("número de árvores B")
ax.set_ylabel("taxa de erro")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

O erro de teste anda entre 0,2282 e 0,2550 ao longo de toda a curva, de 0,2282 × 149 = 34 a 0,2550 × 149 = 38 pacientes errados, e cada degrau da linha laranja são pacientes de teste que trocam de lado, de 1/149 cada. O OOB anda em degraus do mesmo jeito, de 1/148. Com poucas árvores, a previsão OOB de cada paciente junta só cerca de $B/3$ votos, uns 10 com 30 árvores, e a troca de um ou dois votos já muda a previsão. De 100 árvores em diante, o teste fica entre 0,2416 e 0,2550, numa faixa de dois pacientes, de 0,2416 × 149 = 36 a 38: mais árvores não melhoraram o erro de teste. Por isso a comparação entre as duas estimativas se faz no modelo final, de 300 árvores, e não tamanho a tamanho.

Em 300 árvores, o OOB ficou abaixo do teste, e a CV, abaixo dos dois: é o booleano `CV < OOB < teste` do chunk `oob-cv-teste`. Nesta divisão, as duas estimativas feitas no treino saíram otimistas em relação ao teste.

### Uma divisão só não decide

O OOB abaixo do teste pode ser uma regra do método ou depender da divisão, e em 20 divisões diferentes as duas hipóteses preveem coisas diferentes. Se o OOB ficasse abaixo do teste por regra do método, ficaria abaixo nas 20; se a posição dependesse da divisão, os dois lados apareceriam. O chunk refaz a divisão com as sementes 0 a 19, sempre metade para o teste, e ajusta em cada uma o mesmo bagging de 300 árvores:

In [ ]:
divisoes = []
for s in range(20):
    tr_X, te_X, tr_y, te_y = train_test_split(
        X_c, y_c, test_size=0.5, random_state=s, stratify=y_c)
    modelo = RandomForestClassifier(
        n_estimators=300, max_features=None, oob_score=True,
        random_state=13).fit(tr_X, tr_y)
    divisoes.append([1 - modelo.oob_score_,
                     1 - modelo.score(te_X, te_y)])
divisoes = pd.DataFrame(divisoes, columns=["oob", "teste"])
diferenca = divisoes["teste"] - divisoes["oob"]
abaixo, acima = (diferenca > 0).sum(), (diferenca < 0).sum()
print("OOB abaixo do teste:", abaixo, "de 20")
print("OOB acima do teste:", acima, "de 20")
print("cada lado em 6 ou mais:", bool(min(abaixo, acima) >= 6))
print("média de teste − OOB:", numero_br(diferenca.mean(), 3))
print("  menor que 0,01:", abs(diferenca.mean()) < 0.01)
print("média de |teste − OOB|:",
      numero_br(diferenca.abs().mean(), 3))
print("  mais de 7/149:", diferenca.abs().mean() > 7 / 149)

> **🔧 Função**
>
> **`pd.DataFrame(lista, columns)`** — uma tabela a partir de uma lista de linhas, com os nomes das colunas. **`s.abs()`** — o valor absoluto de cada elemento.

In [ ]:
# Figura: O erro OOB (horizontal) e o erro de teste (vertical) do bagging de 300 árvores em 20 divisões do `Heart`, metade para o teste, uma por ponto. Na linha cinza, os dois seriam iguais; acima dela, o OOB ficou abaixo do teste.
fig, ax = plt.subplots(figsize=(3.8, 3.6))
ax.axline((0.1, 0.1), slope=1, color="0.45", linewidth=1.5)
ax.scatter(divisoes["oob"], divisoes["teste"], s=36, color="C0",
           alpha=0.8)
ax.set_xlim(0.10, 0.32)
ax.set_ylim(0.10, 0.32)
ax.set_aspect("equal")
marcas = np.arange(0.10, 0.31, 0.05)
ax.set_xticks(marcas)
ax.set_yticks(marcas)
formato = lambda v, _: f"{v:.2f}".replace(".", ",")
ax.xaxis.set_major_formatter(formato)
ax.yaxis.set_major_formatter(formato)
ax.set_xlabel("erro OOB")
ax.set_ylabel("erro de teste")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.axline(ponto, slope)`** — uma reta que passa pelo `ponto` com a inclinação dada, de borda a borda do gráfico; com inclinação 1, é a diagonal em que os dois eixos são iguais. **`ax.set_aspect("equal")`** — a mesma escala nos dois eixos, para a diagonal ficar a 45 graus.

Os dois lados aparecem, cada um em 6 ou mais das 20 divisões: em algumas o OOB ficou abaixo do teste, em outras acima. A posição do OOB numa divisão isolada não é regra do método. Em média, as diferenças quase se cancelam, e o teste ficou menos de 0,01 acima do OOB. Numa divisão isolada, porém, a distância entre os dois foi, em média, de mais de 7/149, o peso de sete pacientes de teste. Uma tendência de poucos pacientes, se existisse, ficaria escondida atrás dessa distância típica, e a divisão da seção, em que o OOB ficou abaixo do teste, não diz para que lado o OOB erra.

Sim, dá para estimar o erro de teste do bagging sem conjunto de validação e sem dez ajustes: um ajuste de 300 árvores deu um erro OOB entre 0,21 e 0,24, que caiu entre a CV e o teste desta divisão. O OOB e o teste são duas estimativas ruidosas do mesmo erro, cada uma feita com cerca de 150 pacientes. Como o OOB não gasta pacientes, não é preciso reservar um teste para medir o bagging: os 297 podem entrar todos no ajuste, e o OOB continua disponível para escolher entre versões do modelo, como o número de colunas sorteadas por divisão na floresta aleatória da seção 13.3.

Nesse uso vale o cuidado da seção 12.2: o OOB da versão escolhida é o menor de vários erros estimados com os mesmos pacientes, e por isso tende a sair otimista.

## Florestas Aleatórias

> **📌 Nota**
>
> Esta seção corresponde à seção 8.2.2 de James et al. (2023).

No `Hitters`, várias colunas medem quase a mesma coisa: as vezes ao bastão, as rebatidas, as corridas e as bases por bolas acumuladas na carreira crescem juntas com o tempo de jogo de cada atleta. Uma árvore crescida numa amostra bootstrap procura o melhor corte entre todas as colunas, e o melhor tende a sair de uma dessas. Se quase toda árvore do bagging começa pela mesma pergunta, as árvores se parecem, e a média de árvores parecidas reduz menos a variância que a média de árvores diferentes. Dá para obrigar as árvores a serem diferentes umas das outras?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Árvores parecidas demais

Os dados são os jogadores de `Hitters` com salário registrado, com o logaritmo do salário como resposta e as colunas de texto trocadas por indicadoras. O treino tem os mesmos 132 jogadores da seção 13.1, sorteados com a semente 12:

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
y = np.log(hitters["salario"])
X = pd.get_dummies(hitters.drop(columns="salario"),
                   drop_first=True, dtype=float)
treino_X, teste_X, treino_y, teste_y = train_test_split(
    X, y, train_size=132, random_state=12)
print("treino:", len(treino_X), "  teste:", len(teste_X))
print("colunas:", X.shape[1])

O bagging de árvores de regressão é o `RandomForestRegressor` com `max_features=None`, em que toda divisão de toda árvore pode usar qualquer uma das 19 colunas. O chunk ajusta 500 árvores e conta, para cada coluna, quantas árvores a usam na raiz:

In [ ]:
bagging = RandomForestRegressor(n_estimators=500, max_features=None,
                                random_state=13)
bagging.fit(treino_X, treino_y)
raizes_bag = pd.Series([X.columns[arvore.tree_.feature[0]]
                        for arvore in bagging.estimators_]).value_counts()
print(raizes_bag.to_string())
fracao = raizes_bag.iloc[0] / 500
print("primeira, entre 0,4 e 0,5:", 0.4 < fracao < 0.5)
print("três primeiras, mais de 4/5:",
      raizes_bag.iloc[:3].sum() / 500 > 0.8)

> **🔧 Função**
>
> **`RandomForestRegressor(n_estimators, max_features, random_state)`** — ajusta `n_estimators` árvores de regressão sem limite de tamanho, cada uma numa amostra bootstrap do treino, e prevê com a média delas. `max_features` é o número de colunas sorteadas como candidatas em cada divisão; com `None`, todas são candidatas e o modelo é o bagging. **O padrão do regressor, `1.0`, também usa todas as colunas**: sem dizer `max_features`, o `RandomForestRegressor` faz bagging. `random_state` fixa os sorteios.
>
> **`modelo.estimators_`** — a lista das árvores ajustadas. **`arvore.tree_.feature`** — a posição, entre as colunas de `X`, da coluna usada em cada nó da árvore; o nó 0 é a raiz. **`X.columns`** — os nomes das colunas; `X.columns[k]` é o nome da coluna de posição `k`.
>
> **`pd.Series(lista).value_counts()`** — quantas vezes cada valor aparece na lista, do mais frequente ao menos frequente. **`s.to_string()`** — a série inteira como texto, sem a linha de tipo no fim. Com `.iloc`, `raizes_bag.iloc[:3]` pega os três primeiros valores, pela posição.

Só 6 das 19 colunas chegam à raiz de alguma árvore. A primeira da lista, `vezes_ao_bastao_carreira`, está na raiz de quase metade das 500 árvores, e as três primeiras juntas, em mais de quatro de cada cinco. O número exato de árvores de cada coluna pode variar em uma ou duas de um computador para outro: numa amostra bootstrap, dois cortes em colunas diferentes podem ser igualmente bons, ou quase, e o desempate depende do arredondamento.

As árvores do bagging diferem porque as amostras diferem, mas a primeira pergunta, a que divide os jogadores em dois grupos grandes, se repete. Na seção 13.1, a variância da média de $B$ árvores com variância $\sigma^2$ e correlação $\rho$ entre cada par saiu como $\rho\,\sigma^2 + (1 - \rho)\,\sigma^2/B$: a parcela $\rho\,\sigma^2$ não cai com $B$. Árvores que se parecem têm $\rho$ grande, e mais árvores não resolvem.

### Sortear as colunas a cada divisão

A saída é impedir que a mesma coluna forte seja sempre a escolhida, e o jeito de impedir é tirá-la da disputa numa parte das divisões.

> **🔷 Conceito**
>
> A ***floresta aleatória*** (*random forest*) ajusta, como o bagging, uma árvore sem poda em cada uma de $B$ amostras bootstrap e prevê com a média (ou com o voto, em classificação). A diferença está em cada divisão de cada árvore: antes de procurar o melhor corte, sorteiam-se $m$ dos $p$ preditores, e o corte só pode usar um desses $m$. O sorteio é refeito a cada divisão. A escolha usual é $m \approx \sqrt{p}$, e com $m = p$ a floresta é o bagging.

No `Hitters`, $p = 19$ e $\sqrt{19} \approx 4{,}4$, e a floresta usa $m = 4$. Uma coluna fixa fica fora do sorteio de uma divisão com probabilidade $(p - m)/p = (19 - 4)/19 \approx 0{,}79$: em cerca de quatro de cada cinco divisões, `vezes_ao_bastao_carreira` nem é candidata, e outras colunas têm a sua vez. O chunk ajusta a floresta com $m = 4$ no mesmo treino e com a mesma semente:

In [ ]:
floresta = RandomForestRegressor(n_estimators=500, max_features=4,
                                 random_state=13)
floresta.fit(treino_X, treino_y)
raizes_flo = pd.Series([X.columns[arvore.tree_.feature[0]]
                        for arvore in floresta.estimators_]).value_counts()
print("colunas na raiz:")
print("  bagging", len(raizes_bag), "  floresta", len(raizes_flo))
print("primeira da floresta,")
print("  menos de 1/5:", raizes_flo.iloc[0] / 500 < 0.2)

Na floresta, 15 colunas chegam à raiz, contra 6 no bagging, e nenhuma está na raiz de um quinto das árvores. Nas divisões em que a melhor coluna ficou fora do sorteio, a árvore corta por uma coluna mais fraca. Cada árvore da floresta, sozinha, fica melhor ou pior que uma árvore do bagging? O chunk mede o MSE de cada uma das 500 árvores nos 131 jogadores de teste e tira a média, nos dois modelos:

In [ ]:
uma_arvore = {}
for nome, modelo in [("bagging", bagging), ("floresta", floresta)]:
    erros_arvores = [
        mean_squared_error(teste_y, arvore.predict(teste_X.to_numpy()))
        for arvore in modelo.estimators_]
    uma_arvore[nome] = np.mean(erros_arvores)
print("MSE de teste de uma árvore,")
print("  na média das 500, floresta")
print("  acima por mais de 0,01:",
      uma_arvore["floresta"] > uma_arvore["bagging"] + 0.01)

> **🔧 Função**
>
> **`teste_X.to_numpy()`** — a tabela como array, sem os nomes das colunas; as árvores de `estimators_` foram ajustadas sem os nomes, e prever com a tabela emitiria um aviso.

O sorteio tem um custo: o MSE de uma árvore da floresta, na média das 500, fica mais de 0,01 acima do de uma árvore do bagging. A figura põe as duas contagens lado a lado, com as oito colunas que mais aparecem na raiz somando os dois modelos:

In [ ]:
# Figura: Quantas das 500 árvores usam cada coluna na raiz, no treino de 132 jogadores do `Hitters`. Em cima, o bagging, em que toda divisão pode usar as 19 colunas (azul); embaixo, a floresta aleatória com $m = 4$ colunas sorteadas por divisão (verde). As oito colunas são as mais frequentes somando os dois modelos.
colunas = (raizes_bag.add(raizes_flo, fill_value=0)
           .sort_values(ascending=False).index[:8])
fig, eixos = plt.subplots(2, 1, figsize=(3.8, 4.6), sharex=True)
paineis = [(eixos[0], raizes_bag, "C0", "bagging, m = 19"),
           (eixos[1], raizes_flo, "C2", "floresta, m = 4")]
for ax, raizes, cor, titulo in paineis:
    contagem = raizes.reindex(colunas, fill_value=0)
    ax.barh(range(8), contagem.values, color=cor, height=0.7)
    ax.set_yticks(range(8), colunas, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(titulo, loc="left", fontsize=9)
eixos[1].set_xlabel("árvores com a coluna na raiz")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`s.add(outra, fill_value=0)`** — soma duas séries rótulo a rótulo; um rótulo que só existe numa delas conta como 0 na outra. **`s.sort_values(ascending=False)`** — a série em ordem decrescente de valor. **`s.index[:8]`** — os oito primeiros rótulos. **`s.reindex(rotulos, fill_value=0)`** — a série na ordem de `rotulos`, com 0 para o rótulo que ela não tem.
>
> Em **`plt.subplots(2, 1, figsize, sharex=True)`**, os dois primeiros números pedem dois painéis, um sobre o outro, e `sharex=True` dá aos dois o mesmo eixo horizontal; `eixos` é a lista dos painéis. **`ax.barh(posicoes, larguras, color, height)`** — barras horizontais. **`ax.invert_yaxis()`** — põe a primeira barra no alto. **`ax.set_title(titulo, loc="left", fontsize)`** — o título do painel, alinhado à esquerda.

Uma raiz diferente puxa o resto da árvore: os dois grupos em que a raiz divide os jogadores são outros, e as perguntas seguintes também. Uma forma de medir o quanto as árvores de um mesmo modelo se parecem é olhar para as previsões delas nos 131 jogadores do teste: se duas árvores sobem e descem juntas de jogador para jogador, a correlação entre as previsões delas é alta. O chunk ajusta 200 árvores para cada $m$, de 19 (o bagging) a 1, e tira a média da correlação sobre todos os pares de árvores. Antes de rodar: com $m = 1$, em que a coluna de cada divisão é sorteada sem escolha nenhuma, a correlação deve subir ou cair?

In [ ]:
correlacao = {}
for m in [19, 9, 6, 4, 2, 1]:
    modelo = RandomForestRegressor(n_estimators=200, max_features=m,
                                   random_state=13).fit(treino_X, treino_y)
    previsoes = np.array([arvore.predict(teste_X.to_numpy())
                          for arvore in modelo.estimators_])
    pares = np.triu_indices(200, k=1)
    correlacao[m] = np.corrcoef(previsoes)[pares].mean()
    print(f"m = {m:>2}: {numero_br(correlacao[m])}")
valores = list(correlacao.values())
print("cai a cada m menor:",
      all(a > b for a, b in zip(valores, valores[1:])))

> **🔧 Função**
>
> **`np.array(lista)`** — um array a partir de uma lista; aqui, uma linha por árvore e uma coluna por jogador de teste.
>
> **`np.corrcoef(m)`** — a matriz de correlações entre as linhas de `m`, uma linha e uma coluna por árvore. **`np.triu_indices(n, k=1)`** — as posições acima da diagonal de uma matriz $n \times n$, isto é, cada par de árvores uma vez só, sem a árvore com ela mesma.

A correlação cai a cada $m$ menor: de 0,82 no bagging a 0,78 com $m = 4$ e 0,65 com $m = 1$. Mesmo com $m = 1$ as árvores continuam parecidas, porque todas veem os mesmos 132 jogadores e todas tentam prever o mesmo salário. Ao longo dos jogadores, toda árvore razoável acompanha o salário, alto para uns e baixo para outros, e isso sozinho já deixa a correlação alta.

Essa correlação não é o $\rho$ da fórmula. Ela é medida entre as árvores de **um** treino e percorre os 131 jogadores; o $\rho$ fixa um jogador e se mede ao longo de muitos conjuntos de treino. A queda de 0,82 para 0,65 também não isola o quanto as árvores discordam: árvores com mais ruído, cada uma, baixam essa correlação mesmo com o mesmo $\rho$, e uma árvore da floresta, em média, erra mais no teste que uma árvore do bagging, como se mediu acima.

A variância da média cai, então? Quem responde é a medição da seção 13.1: 50 conjuntos de treino de 132 jogadores sorteados entre os 263 e, em cada um, o bagging e a floresta com $m = 4$, de 50 árvores cada, prevendo os jogadores que ficaram de fora.

In [ ]:
posicoes = np.arange(len(X))
dois_modelos = {"bagging": None, "floresta": 4}
previsoes = {nome: np.full((50, len(X)), np.nan) for nome in dois_modelos}
erro_fora = {nome: np.zeros(50) for nome in dois_modelos}
for s in range(50):
    dentro, fora = train_test_split(posicoes, train_size=132,
                                    random_state=s)
    for nome, m in dois_modelos.items():
        modelo = RandomForestRegressor(n_estimators=50, max_features=m,
                                       random_state=13)
        modelo.fit(X.iloc[dentro], y.iloc[dentro])
        previsoes[nome][s, fora] = modelo.predict(X.iloc[fora])
        erro_fora[nome][s] = mean_squared_error(
            y.iloc[fora], previsoes[nome][s, fora])
var = {nome: np.nanvar(p, axis=0) for nome, p in previsoes.items()}
for nome, p in previsoes.items():
    erro_medio = np.nanmean((p - y.values) ** 2, axis=0)
    resto = (np.nanmean(p, axis=0) - y.values) ** 2
    print(f"{nome}: erro {numero_br(erro_medio.mean())}")
    print(f"  variância {numero_br(var[nome].mean(), 3)}"
          f" + resto {numero_br(resto.mean())}")
    print("  soma sem arredondar = erro:",
          np.isclose(var[nome].mean() + resto.mean(),
                     erro_medio.mean()))
menor = (var["floresta"] < var["bagging"]).sum()
print("variância menor na floresta,")
print("  mais de 200 jogadores:", menor > 200)
ganhou = (erro_fora["floresta"] < erro_fora["bagging"]).sum()
print("floresta erra menos,")
print("  em mais de 35 dos 50 treinos:", ganhou > 35)
print("  e não em todos:", ganhou < 50)

> **🔧 Função**
>
> **`np.zeros(n)`** — um array de `n` zeros; aqui, um lugar para o erro de cada conjunto de treino. **`np.isclose(a, b)`** — `True` se `a` e `b` são iguais a menos do arredondamento de ponto flutuante.

A conta é a da seção 13.1: para cada jogador, a variância das previsões e o resto, o quadrado da distância entre o centro delas e o log do salário observado, e as médias sobre os 263 jogadores. A variância média cai de 0,026 no bagging para 0,018 na floresta, uma queda de 0,026 − 0,018 = 0,008, quase um terço de 0,026. O resto fica em 0,20 nos dois, e o erro médio nos jogadores de fora cai de 0,23 para 0,21. As parcelas impressas estão arredondadas, e por isso 0,018 + 0,20 não dá 0,21 na página; sem arredondar, variância mais resto é o erro, nos dois modelos. A variância é menor na floresta em mais de 200 dos 263 jogadores, e a floresta erra menos que o bagging em mais de 35 dos 50 treinos, mas não em todos.

Na divisão da semente 12, uma árvore da floresta, em média, errou mais no teste que uma árvore do bagging; nos 50 treinos, a média da floresta errou menos que a do bagging na maior parte deles. Nessas duas medições, o que a média ganhou por juntar árvores menos parecidas compensou o que uma árvore, em média, perdeu.

### Quantas colunas sortear

Com quantas colunas sorteadas por divisão a floresta erra menos? O $m$ é o parâmetro que a floresta acrescenta ao bagging, e o erro OOB da seção 13.2 permite escolhê-lo sem separar jogadores e sem validação cruzada: um ajuste por valor de $m$. No regressor, o `oob_score_` é o $R^2$, e o MSE OOB sai das previsões guardadas em `oob_prediction_`. O chunk mede, para cada $m$, o MSE OOB nos 132 jogadores de treino e o MSE nos 131 do teste, com 500 árvores:

In [ ]:
erros_m = {}
for m in [19, 9, 6, 4, 2, 1]:
    modelo = RandomForestRegressor(n_estimators=500, max_features=m,
                                   oob_score=True, random_state=13)
    modelo.fit(treino_X, treino_y)
    erros_m[m] = [mean_squared_error(treino_y, modelo.oob_prediction_),
                  mean_squared_error(teste_y, modelo.predict(teste_X))]
erros_m = pd.DataFrame(erros_m, index=["oob", "teste"]).T
print(erros_m.map(lambda v: numero_br(v, 3)).to_string())
oob, teste = erros_m["oob"], erros_m["teste"]
print("OOB de m = 4 abaixo de m = 19")
print("  e de m = 1 por mais de 0,005:",
      oob.loc[4] < min(oob.loc[19], oob.loc[1]) - 0.005)
print("menor OOB em m = 4 ou 6:", oob.idxmin() in (4, 6))
print("menor teste em m =", teste.idxmin())
print("teste entre 0,27 e 0,30:",
      bool(teste.between(0.27, 0.30).all()))
print("de um m a outro, OOB e teste")
print("  variam menos de 0,02:",
      oob.max() - oob.min() < 0.02
      and teste.max() - teste.min() < 0.02)
print("OOB abaixo do teste em todo m:",
      bool((oob < teste).all()))

> **🔧 Função**
>
> **`df.T`** — a tabela transposta, uma linha por $m$. **`df.map(funcao)`** — aplica a função a cada valor; aqui, para escrever os números com vírgula. **`s.between(a, b)`** — `True` onde o valor está entre `a` e `b`. **`s.idxmin()`** — o rótulo do menor valor; aqui, o $m$ de menor erro.
>
> O `pd.DataFrame` recebe aqui um dicionário: cada chave, um $m$, vira uma coluna, e `index` dá o nome das linhas; por isso a tabela sai deitada e o `.T` a transpõe.

O MSE OOB vai de 0,167 com $m = 19$, o bagging, a 0,159 com $m = 4$, e fica mais de 0,005 abaixo do OOB das duas pontas, $m = 19$ e $m = 1$. O menor OOB está em $m = 4$ ou em $m = 6$: pelo OOB, a escolha seria um desses dois. O teste, nos 131 jogadores de fora, põe o menor erro em $m = 2$. As duas medidas variam pouco de um $m$ para outro, menos de 0,02 nas duas colunas, e com diferenças desse tamanho a ordem entre os $m$ do meio muda de uma medida para a outra. O OOB do $m$ escolhido é também o menor de seis estimativas feitas com os mesmos jogadores, com o otimismo de que a seção 13.2 falou.

In [ ]:
# Figura: O MSE da floresta aleatória de 500 árvores no `Hitters` contra o número $m$ de preditores sorteados por divisão, de 1 a 19; $m = 19$ é o bagging. Em roxo tracejado, o MSE OOB nos 132 jogadores de treino; em cinza, o MSE nos 131 jogadores de teste. Aqui as cores separam as duas medidas de erro, não o bagging e a floresta. Os valores de $m$ estão igualmente espaçados no eixo.
ordem = sorted(erros_m.index)
fig, ax = plt.subplots(figsize=(3.8, 3.2))
ax.plot(range(6), erros_m.loc[ordem, "oob"], color="C3", marker="o",
        linestyle="--", label="OOB")
ax.plot(range(6), erros_m.loc[ordem, "teste"], color="0.45", marker="o",
        label="teste")
ax.set_xticks(range(6), [str(m) for m in ordem])
ax.set_ylim(0.10, 0.35)
ax.set_xlabel("preditores por divisão, m")
ax.set_ylabel("MSE")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.2f}".replace(".", ","))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

Na figura, `erros_m.loc[ordem, "teste"]` usa o `.loc` com dois argumentos: as linhas de `ordem`, de $m = 1$ a $m = 19$, e a coluna `teste`.

O OOB fica abaixo do teste em todos os $m$. Os dois medem o erro em jogadores diferentes: o OOB, nos 132 do treino, cada um previsto pelas árvores que não o sortearam; o teste, nos 131 que ficaram de fora. Com uma divisão só, um grupo pode sair mais fácil de prever que o outro, e a distância de nível vale para todo $m$, sem dizer nada sobre qual $m$ é melhor. O que se lê da figura é a ordem dentro de cada curva, e ela não é a mesma nas duas. Os 50 conjuntos de treino medidos antes, que trocam de divisão a cada vez, dão o sinal mais firme sobre $m = 4$ contra o bagging: a floresta errou menos em mais de 35 deles.

### Bagging contra floresta nos pacientes

Em classificação, a comparação se faz do mesmo jeito. Os pacientes do `Heart` estão divididos como na seção 13.2: metade para o teste, sorteada com a semente 13, e as colunas de texto trocadas por indicadoras de todas as categorias.

In [ ]:
coracao = pd.read_csv("dados/Heart.csv").dropna()
y_c = coracao["doenca_cardiaca"]
X_c = pd.get_dummies(coracao.drop(columns="doenca_cardiaca"),
                     dtype=float)
treino_Xc, teste_Xc, treino_yc, teste_yc = train_test_split(
    X_c, y_c, test_size=0.5, random_state=13, stratify=y_c)
print("treino:", len(treino_Xc), "  teste:", len(teste_Xc))
print("colunas:", X_c.shape[1])

Com 18 colunas, $\sqrt{18} \approx 4{,}2$, e a floresta sorteia 4 colunas por divisão: o `max_features="sqrt"`, que é o padrão do `RandomForestClassifier`. O chunk faz crescer o bagging e a floresta de 10 em 10 árvores, de 20 a 300, e mede o erro OOB e o de teste em cada tamanho. Com 4 colunas por divisão em vez de 18, a floresta deve errar menos em todo tamanho, ou só a partir de muitas árvores?

In [ ]:
tamanhos = range(20, 301, 10)
curvas = {}
for nome, m in [("bagging", None), ("floresta", "sqrt")]:
    modelo = RandomForestClassifier(n_estimators=20, max_features=m,
                                    oob_score=True, warm_start=True,
                                    random_state=13)
    curva = pd.DataFrame(index=tamanhos, columns=["oob", "teste"],
                         dtype=float)
    for B in tamanhos:
        modelo.set_params(n_estimators=B).fit(treino_Xc, treino_yc)
        curva.loc[B] = [1 - modelo.oob_score_,
                        1 - modelo.score(teste_Xc, teste_yc)]
    curvas[nome] = curva
print("m da floresta:", modelo.estimators_[0].max_features_)
for nome, curva in curvas.items():
    print(f"{nome}, B = 300:")
    print(f"  teste {numero_br(curva['teste'].loc[300], 4)}"
          f" ({round(curva['teste'].loc[300] * 149)} erros)")
distancia = curvas["bagging"]["teste"] - curvas["floresta"]["teste"]
pacientes = (distancia * 149).round()
print("floresta com 2 ou mais erros")
print("  a menos:", (pacientes >= 2).sum(), "de", len(pacientes))
print("floresta com mais erros:", (pacientes < 0).sum())
oob_300 = [curvas[n]["oob"].loc[300] for n in curvas]
print("OOB em 300, floresta abaixo")
print("  por mais de 5/148:", oob_300[0] - oob_300[1] > 5 / 148)

> **🔧 Função**
>
> **`arvore.max_features_`** — o número de colunas sorteadas em cada divisão daquela árvore, já convertido de `"sqrt"` para um inteiro. **`s.round()`** — arredonda cada valor para o inteiro mais próximo; aqui, a distância entre as taxas de erro vezes 149 vira um número de pacientes.

Com 300 árvores, o bagging erra 37 dos 149 pacientes de teste e a floresta, 32, cinco pacientes a menos (37 − 32 = 5). Em 28 dos 29 tamanhos a floresta erra pelo menos dois pacientes a menos que o bagging, e em nenhum erra mais. Em 300 árvores, o erro OOB da floresta também fica abaixo do OOB do bagging, por mais de cinco pacientes de treino.

In [ ]:
# Figura: Taxa de erro nos pacientes do `Heart` contra o número de árvores $B$, do bagging (azul) e da floresta aleatória com 4 colunas sorteadas por divisão (verde). A linha contínua é o erro nos 149 pacientes de teste; a tracejada, o erro OOB nos 148 de treino.
fig, ax = plt.subplots(figsize=(3.8, 3.6))
for nome, cor in [("bagging", "C0"), ("floresta", "C2")]:
    ax.plot(curvas[nome].index, curvas[nome]["teste"], color=cor,
            label=f"{nome}, teste")
    ax.plot(curvas[nome].index, curvas[nome]["oob"], color=cor,
            linestyle="--", label=f"{nome}, OOB")
ax.set_xlim(0, 320)
ax.set_ylim(0.15, 0.30)
ax.set_yticks(np.arange(0.15, 0.31, 0.05))
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.2f}".replace(".", ","))
ax.set_xlabel("número de árvores B")
ax.set_ylabel("taxa de erro")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

Cinco pacientes de 149 são uma diferença de 5/149 = 0,034 na taxa de erro, e outra divisão dos 297 pacientes daria outros erros aos dois modelos. O chunk refaz a comparação em 20 divisões, com as sementes 0 a 19, cada uma com 200 árvores por modelo, e conta os erros de teste:

In [ ]:
erros = []
for s in range(20):
    tr_X, te_X, tr_y, te_y = train_test_split(
        X_c, y_c, test_size=0.5, random_state=s, stratify=y_c)
    erros.append([
        (RandomForestClassifier(n_estimators=200, max_features=m,
                                random_state=13).fit(tr_X, tr_y)
         .predict(te_X) != te_y).sum()
        for m in [None, "sqrt"]])
erros = pd.DataFrame(erros, columns=["bagging", "floresta"])
vantagem = erros["bagging"] - erros["floresta"]
print("floresta erra menos:", (vantagem > 0).sum(), "de 20")
print("empate:", (vantagem == 0).sum(), "de 20")
print("bagging erra menos:", (vantagem < 0).sum(), "de 20")
print("vantagem média da floresta,")
print("  entre 2 e 4 pacientes:", 2 < vantagem.mean() < 4)
print("maior vantagem do bagging,")
print("  mais de 2 pacientes:", vantagem.min() < -2)

> **🔧 Função**
>
> **`(previsto != te_y).sum()`** — o número de pacientes de teste em que a previsão difere da resposta, isto é, os erros.

A floresta errou menos em 16 das 20 divisões, empatou em 1 e errou mais em 3. Em média, errou entre dois e quatro pacientes a menos, e em ao menos uma divisão o bagging ganhou por mais de dois pacientes. No `Heart`, como no `Hitters`, sortear as colunas ajudou na maior parte das divisões, sem ganhar em todas.

Sortear as colunas troca árvores melhores por árvores menos parecidas. Um $m$ pequeno costuma ajudar quando há muitos preditores correlacionados entre si, como as colunas de carreira do `Hitters`, cujas correlações a seção 12.5 mediu: quando `vezes_ao_bastao_carreira` fica fora do sorteio, outra coluna de carreira quase igual a ela pode entrar no lugar. Um $m$ pequeno pode atrapalhar, num caso não medido aqui, quando há poucas colunas com sinal entre muitas de ruído: muitas divisões ficam só com colunas de ruído para escolher [Hastie et al. (2009), seção 15.3.4]. Por isso o $m$ é um parâmetro a escolher, e $m = p$, o bagging, é um dos valores da escolha.

## Boosting

> **📌 Nota**
>
> Esta seção corresponde à seção 8.2.3 de James et al. (2023).

No bagging e na floresta aleatória, cada árvore é ajustada à sua amostra bootstrap sem saber o que as outras fizeram, e as árvores só se encontram na média do fim. Um jogador cujo salário todas as árvores erram para cima continua errado para cima na média. E se as árvores fossem ajustadas uma depois da outra, e cada árvore nova olhasse só para o que as anteriores ainda erram, corrigindo um pouco?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, train_test_split

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Aprender devagar

Os dados são os jogadores de `Hitters` com salário registrado, com o logaritmo do salário como resposta, as colunas de texto trocadas por indicadoras e os mesmos 132 jogadores de treino da seção 13.1:

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
y = np.log(hitters["salario"])
X = pd.get_dummies(hitters.drop(columns="salario"),
                   drop_first=True, dtype=float)
treino_X, teste_X, treino_y, teste_y = train_test_split(
    X, y, train_size=132, random_state=12)
print("treino:", len(treino_X), "  teste:", len(teste_X))

Uma árvore grande ajustada de uma vez ao log do salário tenta acertar tudo numa só passada e, como a seção 12.2 mostrou, acompanha o ruído do treino. O boosting faz o contrário: começa de um modelo que prevê zero para todo mundo e o melhora aos poucos. A cada passo, calcula o resíduo de cada jogador, o quanto o modelo atual ainda erra, ajusta uma árvore pequena a esses resíduos e soma ao modelo só uma fração dela.

> **🔷 Conceito**
>
> O ***boosting*** de árvores de regressão ajusta $B$ árvores em sequência, cada uma aos resíduos deixados pelas anteriores. Com $d$, o número de divisões de cada árvore, e $\lambda$, um número positivo pequeno:
>
> 1. Comece com $\hat f(x) = 0$ e com os resíduos $r_i = y_i$ para todo $i$ do treino.
> 2. Para $b = 1, 2, \dots, B$:
>    a. ajuste uma árvore $\hat f^b$ com $d$ divisões, isto é, $d + 1$ folhas, tendo $r$, e não $y$, como resposta;
>    b. some ao modelo a árvore encolhida por $\lambda$,
>    $$\hat f(x) \leftarrow \hat f(x) + \lambda \hat f^b(x);$$
>    c. atualize os resíduos,
>    $$r_i \leftarrow r_i - \lambda \hat f^b(x_i).$$
> 3. O modelo final é a soma das árvores encolhidas,
> $$
> \hat f(x) = \sum_{b=1}^{B} \lambda \hat f^b(x).
> $$
>
> O parâmetro $\lambda$ é a ***taxa de aprendizado*** (*shrinkage*, ou *learning rate*): quanto de cada árvore entra no modelo.

Não há amostra bootstrap: toda árvore vê os 132 jogadores, com uma resposta diferente a cada passo.

No `scikit-learn`, o algoritmo é o `GradientBoostingRegressor`. Com `init="zero"`, ele começa de $\hat f(x) = 0$, como no passo 1. O chunk ajusta $B = 1.000$ árvores de uma divisão só, $d = 1$, com $\lambda = 0{,}01$, mostra a primeira delas e calcula a média do log do salário de cada lado do corte dela:

In [ ]:
boost = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.01,
                                  max_leaf_nodes=2, max_depth=None,
                                  init="zero", random_state=13)
boost.fit(treino_X, treino_y)
primeira = boost.estimators_[0, 0]
coluna = X.columns[primeira.tree_.feature[0]]
corte = primeira.tree_.threshold[0]
print("raiz:", coluna)
print("corte:", numero_br(corte, 0))
folhas = primeira.tree_.value[1:, 0, 0]
print("folhas:", numero_br(folhas[0]), "e", numero_br(folhas[1]))
abaixo = treino_X[coluna] <= corte
print("médias de cada lado:", numero_br(treino_y[abaixo].mean()),
      "e", numero_br(treino_y[~abaixo].mean()))
print("média do log do salário:", numero_br(treino_y.mean()))

> **🔧 Função**
>
> **`GradientBoostingRegressor(...)`** — o boosting de árvores de regressão. Os parâmetros usados aqui:
>
> - `n_estimators`: o número de árvores, $B$.
> - `learning_rate`: a taxa $\lambda$.
> - `max_leaf_nodes`: o número máximo de folhas de cada árvore; `max_leaf_nodes=2` dá árvores de uma divisão, $d = 1$.
> - `max_depth`: a profundidade máxima de cada árvore, que vale junto com `max_leaf_nodes`. O padrão, 3, limita cada árvore a 8 folhas; com `None`, só `max_leaf_nodes` limita, qualquer que seja $d$.
> - `init`: com `"zero"`, o modelo começa de $\hat f(x) = 0$; sem ele, começa da média de $y$ no treino.
> - `random_state`: fixa os desempates.
>
> **`modelo.estimators_[b, 0]`** — a árvore de número `b`, contada a partir de 0; `estimators_` é uma tabela de árvores com uma linha por passo e uma coluna só, na regressão. **`arvore.tree_.threshold[0]`** — o valor de corte da raiz. **`arvore.tree_.value[1:, 0, 0]`** — a previsão das folhas de uma árvore de uma divisão: os nós 1 e 2, depois da raiz, que é o nó 0.

A primeira árvore divide os jogadores pelo número de vezes ao bastão na carreira. Como o modelo começa de zero, os resíduos do primeiro passo são o próprio log do salário, e as folhas são a média do log do salário de cada lado do corte: as duas linhas impressas coincidem, 5,11 e 6,46. Com $\lambda = 0{,}01$, só um centésimo disso entra no modelo: depois do primeiro passo, $\hat f(x)$ vale 0,01 × 5,11 ≈ 0,05 para os jogadores de um lado e 0,01 × 6,46 ≈ 0,06 para os do outro, longe da média de 5,91.

Cada passo tira do resíduo de cada jogador um centésimo da previsão da árvore. Antes de rodar o próximo chunk, um palpite: partindo de zero, depois de quantas árvores a previsão média no treino chega perto de 5,91, a média do log do salário? Dez? Cem? Mil? O chunk imprime a previsão média e o MSE de treino depois de 1, 10, 100 e 1.000 árvores:

In [ ]:
previsoes = list(boost.staged_predict(treino_X))
print("no treino, depois de B árvores:")
for b in [1, 10, 100, 1000]:
    print(f"  B = {numero_br(b, 0)}:"
          f" média {numero_br(previsoes[b - 1].mean())},"
          f" MSE {numero_br(boost.train_score_[b - 1])}")
medias = np.array([previsto.mean() for previsto in previsoes])
chega = (medias.round(2) >= round(treino_y.mean(), 2)).argmax() + 1
print("média chega a", numero_br(treino_y.mean()),
      "em B =", numero_br(chega, 0))

> **🔧 Função**
>
> **`modelo.staged_predict(X)`** — as previsões do modelo depois de cada árvore, uma de cada vez: primeiro com a árvore 1, depois com as árvores 1 e 2, e assim por diante até $B$. Um ajuste só dá a curva inteira contra $B$. **`modelo.train_score_`** — no regressor, o MSE de treino depois de cada árvore.

Dez árvores levam a previsão média a 0,57, e cem, a 3,75. A última linha procura a primeira árvore depois da qual a média, arredondada, chega a 5,91: é a de número 696. O MSE de treino é a média dos quadrados dos resíduos, e cai junto, de 34,97 depois da primeira árvore para 0,09 depois da milésima.

O modelo final é a soma das árvores encolhidas. O chunk confere isso nos jogadores de teste, somando as previsões das mil árvores multiplicadas por 0,01, e ajusta o mesmo boosting começando da média, o `init` padrão:

In [ ]:
arvores = boost.estimators_[:, 0]
soma = sum(0.01 * arvore.predict(teste_X.to_numpy()) for arvore in arvores)
print("soma = predict:", np.allclose(soma, boost.predict(teste_X)))
print("raízes das três primeiras:")
for arvore in arvores[:3]:
    print("  ", X.columns[arvore.tree_.feature[0]])
da_media = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.01,
                                     max_leaf_nodes=2, max_depth=None,
                                     random_state=13)
da_media.fit(treino_X, treino_y)
erro_zero = mean_squared_error(teste_y, boost.predict(teste_X))
erro_media = mean_squared_error(teste_y, da_media.predict(teste_X))
print("MSE de teste, de zero:", numero_br(erro_zero, 4))
print("MSE de teste, da média:", numero_br(erro_media, 4))

> **🔧 Função**
>
> **`np.allclose(a, b)`** — `True` se os dois arrays são iguais posição a posição, a menos do arredondamento de ponto flutuante.

O `sum` soma os arrays de previsões posição a posição, um jogador de cada vez, e a soma das mil árvores encolhidas é a previsão do modelo, como no passo 3. As três primeiras árvores cortam pela mesma coluna: depois de poucos passos de um centésimo, os resíduos ainda se parecem com o próprio log do salário, e o melhor corte continua o mesmo. Começar da média poupa o caminho de zero até o nível dos salários. Com $\lambda = 0{,}01$ e mil árvores, esse caminho já foi percorrido, e os dois erros de teste impressos coincidem nas quatro casas.

### Três parâmetros: B, λ e d

O boosting tem três parâmetros: o número de árvores $B$, a taxa $\lambda$ e o número de divisões $d$ de cada árvore. Com $\lambda$ pequeno, cada árvore muda pouco o modelo, e são precisas mais árvores para chegar ao mesmo lugar. Mais árvores sempre melhoram?

Daqui em diante, os modelos partem da média do log do salário no treino, o padrão do `scikit-learn`. Com $\lambda$ pequeno, o ponto de partida pesa, e o chunk mede isso no fim. Para ver o efeito de $B$ e de $\lambda$ juntos, o chunk ajusta, com árvores de uma divisão, 3.000 árvores para cada $\lambda$ em 0,001, 0,01 e 0,1, e mede o MSE de teste depois de cada árvore; no fim, ajusta de novo o $\lambda = 0{,}001$ partindo de zero:

In [ ]:
curvas = {}
modelos = {}
for taxa in [0.001, 0.01, 0.1]:
    modelo = GradientBoostingRegressor(n_estimators=3000, learning_rate=taxa,
                                       max_leaf_nodes=2, max_depth=None,
                                       random_state=13)
    modelo.fit(treino_X, treino_y)
    curvas[taxa] = np.array([mean_squared_error(teste_y, previsto)
                             for previsto in modelo.staged_predict(teste_X)])
    modelos[taxa] = modelo
for taxa, curva in curvas.items():
    print(f"λ = {numero_br(taxa, 3).rstrip('0')}:")
    print(f"  em B = 1: {numero_br(curva[0], 4)}")
    print(f"  menor {numero_br(curva.min(), 4)}"
          f" em B = {numero_br(np.argmin(curva) + 1, 0)}")
    print(f"  em B = 3.000: {numero_br(curva[-1], 4)}")
treino_001 = modelos[0.01].train_score_
print("λ = 0,01, treino em 3.000:", numero_br(treino_001[-1], 4))
print("treino nunca sobe, nos três:",
      all(np.all(np.diff(m.train_score_) <= 0) for m in modelos.values()))
de_zero = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.001,
                                    max_leaf_nodes=2, max_depth=None,
                                    init="zero", random_state=13)
de_zero.fit(treino_X, treino_y)
menor_zero = min(mean_squared_error(teste_y, previsto)
                 for previsto in de_zero.staged_predict(teste_X))
print("λ = 0,001 de zero, menor:", numero_br(menor_zero, 4))

> **🔧 Função**
>
> **`np.argmin(v)`** — a posição do menor valor de `v`, contada a partir de 0; somando 1, o número de árvores. **`np.diff(v)`** — as diferenças entre valores vizinhos de `v`; `np.diff(v) <= 0` em todas as posições quer dizer que `v` nunca sobe. **`np.all(m)`** — `True` se todos os valores de `m` são `True`.

In [ ]:
# Figura: MSE de teste do boosting de árvores de uma divisão no `Hitters`, partindo da média, contra o número de árvores $B$, em escala logarítmica, para três taxas de aprendizado: $\lambda = 0{,}001$ (azul), $0{,}01$ (laranja) e $0{,}1$ (verde). O x marca o menor erro de teste de cada curva. Em cinza tracejado, o MSE de treino com $\lambda = 0{,}01$. O MSE é medido no log do salário.
estagios = np.arange(1, 3001)
fig, ax = plt.subplots(figsize=(3.8, 3.4))
for taxa, cor in [(0.001, "C0"), (0.01, "C1"), (0.1, "C2")]:
    ax.plot(estagios, curvas[taxa], color=cor,
            label=f"λ = {numero_br(taxa, 3).rstrip('0')}")
    melhor = np.argmin(curvas[taxa])
    ax.plot(estagios[melhor], curvas[taxa][melhor], marker="x",
            color=cor, markersize=8, label="_nolegend_")
ax.plot(estagios, treino_001, color="0.45", linestyle="--",
        label="treino, λ = 0,01")
ax.set_xscale("log")
ax.set_xticks([1, 10, 100, 1000], ["1", "10", "100", "1.000"])
ax.set_xlim(0.8, 4000)
ax.set_ylim(0, 0.9)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1f}".replace(".", ","))
ax.set_xlabel("número de árvores B")
ax.set_ylabel("MSE (log do salário)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

Com $\lambda = 0{,}1$, o menor erro de teste chega em $B = 51$; com $\lambda = 0{,}01$, em $B = 635$; com $\lambda = 0{,}001$, o menor erro está na última das 3.000 árvores. Quanto menor $\lambda$, mais árvores foram precisas até o menor erro, nos três valores de $\lambda$ testados. Os dois menores erros, 0,2856 com $\lambda = 0{,}1$ e 0,2851 com $\lambda = 0{,}01$, diferem em 0,0005; com $\lambda = 0{,}001$, 3.000 árvores não chegaram lá, e o menor erro ficou em 0,3027. Partindo de zero, o menor erro de teste do mesmo $\lambda = 0{,}001$ é 0,3931, contra 0,3027 partindo da média: é o caso em que o ponto de partida pesa.

Na figura, a curva verde desce primeiro e sobe primeiro; a laranja faz o mesmo movimento mais à direita, e a azul ainda está descendo quando as 3.000 árvores acabam. Um $\lambda$ menor desloca a curva para a direita no eixo de $B$.

Depois do mínimo, o erro de teste sobe. Com $\lambda = 0{,}01$, ele vai de 0,2851 em $B = 635$ a 0,3126 em $B = 3.000$; com $\lambda = 0{,}1$, de 0,2856 a 0,3832. O erro de treino, enquanto isso, nunca sobe nos três modelos, e com $\lambda = 0{,}01$ termina em 0,0502.

O erro de teste subindo é compatível com correções que, depois do mínimo, acompanham o que é particular desses 132 jogadores. No boosting, um $B$ grande demais pode sobreajustar, ao contrário do bagging, em que uma árvore a mais só entra numa média.

Se o menor erro quase não muda entre $\lambda = 0{,}1$ e $\lambda = 0{,}01$, o que o $\lambda$ menor compra? O chunk marca, para cada $\lambda$, os valores de $B$ com erro de teste a até 5% do menor erro da curva, o vale da curva, e imprime quantos são, o primeiro e o último, a razão entre o último e o primeiro e se a última das 3.000 árvores está no vale:

In [ ]:
folga = 1.05
for taxa, curva in curvas.items():
    vale = np.flatnonzero(curva <= folga * curva.min()) + 1
    print(f"λ = {numero_br(taxa, 3).rstrip('0')}:"
          f" {numero_br(len(vale), 0)} valores de B")
    print(f"  de {numero_br(vale[0], 0)} a {numero_br(vale[-1], 0)},"
          f" razão {numero_br(vale[-1] / vale[0], 1)}")
    print("  último B no vale:", vale[-1] == 3000)

> **🔧 Função**
>
> **`np.flatnonzero(m)`** — as posições em que o array booleano `m` vale `True`, em ordem crescente, contadas a partir de 0; somando 1, os números de árvores.

Com $\lambda = 0{,}1$, 157 valores de $B$ ficam a até 5% do menor erro, de 30 a 187; com $\lambda = 0{,}01$, 1.473, de 315 a 1.787. Contado em árvores, o vale laranja é muito mais largo, mas cada árvore laranja entra no modelo com um décimo do peso de uma verde, e o mesmo trecho do caminho ocupa mais árvores. Medido em proporção, o vale quase não muda: o último $B$ do vale é 6,2 vezes o primeiro com $\lambda = 0{,}1$ e 5,7 vezes com $\lambda = 0{,}01$. Com $\lambda = 0{,}001$, a última das 3.000 árvores ainda está no vale, e ele não fechou.

Nestes dados, o $\lambda$ menor baixou o menor erro só em 0,0005 e não alargou o vale em proporção. Ele anda com um passo mais fino, de um centésimo por árvore, e cobra isso em árvores: o menor erro de teste veio com 635 árvores com $\lambda = 0{,}01$, contra 51 com $\lambda = 0{,}1$, e 635 ÷ 51 ≈ 12,5.

O terceiro parâmetro, $d$, é o tamanho de cada árvore. Uma árvore de uma divisão, $d = 1$, é um ***toco*** (*stump*). Cada toco usa uma coluna só, e uma soma de tocos é uma soma de funções de uma coluna cada: o modelo é ***aditivo***, sem interação entre as colunas.

Com $d = 2$, uma árvore pode dividir por uma coluna e depois, num dos lados, por outra, e o efeito da segunda passa a depender do valor da primeira. Como $d$ divisões usam no máximo $d$ colunas, $d$ controla a ordem das interações que o modelo pode capturar, e é chamado de ***profundidade de interação***. O chunk ajusta, com $\lambda = 0{,}01$ e 3.000 árvores, o boosting com $d = 2$ e $d = 4$, compara com os tocos já ajustados e mostra as duas divisões da primeira árvore de $d = 2$:

In [ ]:
por_d = {1: modelos[0.01]}
for d in [2, 4]:
    por_d[d] = GradientBoostingRegressor(n_estimators=3000,
                                         learning_rate=0.01,
                                         max_leaf_nodes=d + 1,
                                         max_depth=None,
                                         random_state=13)
    por_d[d].fit(treino_X, treino_y)
for d, modelo in por_d.items():
    curva = [mean_squared_error(teste_y, previsto)
             for previsto in modelo.staged_predict(teste_X)]
    folhas = {int(arvore.get_n_leaves()) for arvore in modelo.estimators_[:, 0]}
    print(f"d = {d}: folhas por árvore {folhas}")
    print(f"  menor teste {numero_br(min(curva), 4)}"
          f" em B = {numero_br(np.argmin(curva) + 1, 0)}")
    print(f"  treino em 3.000: {numero_br(modelo.train_score_[-1], 4)}")
raizes = {X.columns[arvore.tree_.feature[0]]
          for arvore in por_d[1].estimators_[:, 0]}
print("colunas usadas pelos tocos:", len(raizes), "de", X.shape[1])
arvore_d2 = por_d[2].estimators_[0, 0].tree_
esquerda = arvore_d2.children_left[0]
lado = "≤" if arvore_d2.feature[esquerda] >= 0 else ">"
segundo = esquerda if lado == "≤" else arvore_d2.children_right[0]
print("1.ª árvore, d = 2, raiz:")
print("  ", X.columns[arvore_d2.feature[0]], "≤",
      numero_br(arvore_d2.threshold[0], 0))
print("lado", lado, "da raiz:")
print("  ", X.columns[arvore_d2.feature[segundo]], "≤",
      numero_br(arvore_d2.threshold[segundo], 1))

> **🔧 Função**
>
> **`arvore.tree_.children_left`**, **`arvore.tree_.children_right`** — para cada nó, o número do nó filho com os jogadores que ficam abaixo ou no corte (esquerda) e acima dele (direita). **`arvore.tree_.feature`** vale um número negativo numa folha, e por isso `feature[no] >= 0` quer dizer que o nó divide.

Com `max_leaf_nodes=d + 1` e `max_depth=None`, toda árvore tem $d + 1$ folhas: 2, 3 e 5. Os 3.000 tocos, juntos, usam 18 das 19 colunas, mas cada toco usa só uma. Na primeira árvore de $d = 2$, os jogadores com até 1.452 vezes ao bastão na carreira são divididos de novo por `impulsionadas_carreira`, e os de mais de 1.452 não: o efeito de `impulsionadas_carreira` nessa árvore existe só de um lado do primeiro corte. É uma interação entre as duas colunas, que nenhuma soma de tocos representa.

Os menores erros de teste ficam em 0,2851 com $d = 1$, 0,2775 com $d = 2$ e 0,2861 com $d = 4$; entre $d = 1$ e $d = 2$, a distância é de 0,2851 − 0,2775 = 0,0076, medida em 131 jogadores de uma divisão só, e não basta para dizer que as interações ajudam.

O que muda claramente é o treino: depois de 3.000 árvores, o MSE de treino cai de 0,0502 com tocos para 0,0067 com $d = 2$ e 0,0002 com $d = 4$. Árvores maiores decoram o treino mais depressa, e o valor de $d$, como o de $B$ e o de $\lambda$, é para escolher com dados de fora do ajuste.

### Escolher B por validação cruzada

O $B$ de menor erro de teste, 635 com $\lambda = 0{,}01$ e tocos, foi escolhido olhando os jogadores de teste, e por isso o erro de teste nesse $B$, 0,2851, é otimista como estimativa do erro em jogadores novos. O $B$ se escolhe no treino, por validação cruzada. Um ajuste de 3.000 árvores por grupo basta: o `staged_predict` dá o erro do grupo de validação depois de cada árvore. O chunk usa 5 grupos, com $\lambda = 0{,}01$ e tocos, e tira a média das cinco curvas:

In [ ]:
grupos = KFold(n_splits=5, shuffle=True, random_state=13)
curva_cv = np.zeros(3000)
for dentro, fora in grupos.split(treino_X):
    modelo = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.01,
                                       max_leaf_nodes=2, max_depth=None,
                                       random_state=13)
    modelo.fit(treino_X.iloc[dentro], treino_y.iloc[dentro])
    curva_cv += np.array([
        mean_squared_error(treino_y.iloc[fora], previsto)
        for previsto in modelo.staged_predict(treino_X.iloc[fora])]) / 5
B_cv = np.argmin(curva_cv) + 1
teste_001 = curvas[0.01]
B_teste = np.argmin(teste_001) + 1
print("B escolhido pela CV:", numero_br(B_cv, 0))
print("MSE de CV nesse B:", numero_br(curva_cv[B_cv - 1], 4))
print(f"MSE de CV em B = {numero_br(B_teste, 0)}:",
      numero_br(curva_cv[B_teste - 1], 4))
print("teste nesse B:", numero_br(teste_001[B_cv - 1], 4))
print("menor teste:", numero_br(teste_001.min(), 4),
      "em", numero_br(B_teste, 0))
entre = curva_cv[min(B_cv, B_teste) - 1:max(B_cv, B_teste)]
print("CV entre os dois B:", numero_br(entre.min(), 4),
      "a", numero_br(entre.max(), 4))
print("CV abaixo do teste em todo B:", np.all(curva_cv < teste_001))
print("em B = 1: CV", numero_br(curva_cv[0], 4),
      "teste", numero_br(teste_001[0], 4))

> **🔧 Função**
>
> **`grupos.split(X)`** — as posições das linhas de cada divisão da validação cruzada: a cada volta do laço, `dentro` tem as posições do ajuste e `fora`, as do grupo de validação.

In [ ]:
# Figura: MSE de validação cruzada de 5 grupos no treino (azul) e MSE de teste (laranja) do boosting de tocos com $\lambda = 0{,}01$, contra o número de árvores $B$, em escala logarítmica. O x azul marca o $B$ escolhido pela validação cruzada; o x laranja, o $B$ de menor erro de teste. O MSE é medido no log do salário.
fig, ax = plt.subplots(figsize=(3.8, 3.4))
ax.plot(estagios, curva_cv, color="C0", label="validação cruzada")
ax.plot(estagios, teste_001, color="C1", label="teste")
ax.plot(B_cv, curva_cv[B_cv - 1], marker="x", color="C0",
        markersize=8, label="_nolegend_")
ax.plot(B_teste, teste_001[B_teste - 1], marker="x", color="C1",
        markersize=8, label="_nolegend_")
ax.set_xscale("log")
ax.set_xticks([1, 10, 100, 1000], ["1", "10", "100", "1.000"])
ax.set_xlim(0.8, 4000)
ax.set_ylim(0, 0.9)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1f}".replace(".", ","))
ax.set_xlabel("número de árvores B")
ax.set_ylabel("MSE (log do salário)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

A validação cruzada escolhe $B = 1.307$. Nesse $B$, o erro nos 131 jogadores de teste é 0,2919, acima do menor erro de teste, 0,2851 em $B = 635$, por 0,2919 − 0,2851 = 0,0068. A CV escolhe com os 132 jogadores de treino, e o $B$ que acerta melhor esses 131 jogadores de teste só é conhecido depois de olhar para eles. Entre os dois valores de $B$, a curva de CV vai de 0,1957 a 0,2000: em $B = 635$ ela vale 0,1998, e em $B = 1.307$, 0,1957, uma diferença de 0,1998 − 0,1957 = 0,0041.

O MSE de CV no $B$ escolhido, 0,1957, fica bem abaixo do erro de teste nesse $B$, 0,2919: são 0,2919 − 0,1957 = 0,0962, e quase toda essa distância vem da divisão. A curva de CV fica abaixo da de teste em todo $B$, inclusive em $B = 1$, antes de qualquer escolha: 0,7414 contra 0,8305. Em $B = 635$, um valor que a CV não escolheu, ela vale 0,1998, e o teste, 0,2851, uma distância de 0,2851 − 0,1998 = 0,0853. Com uma divisão só, os 131 jogadores de teste podem ser mais difíceis de prever que os 132 de treino: a seção 13.3 viu, nessa mesma divisão, o erro OOB também abaixo do de teste. O otimismo de ficar com o menor de 3.000 erros estimados com os mesmos 132 jogadores, que a seção 12.2 viu na poda, existe, mas pesa pouco aqui: ficar com o mínimo baixou a CV em 0,0041 em relação a $B = 635$.

Em classificação, o boosting segue a mesma ideia, cada árvore corrigindo o que as anteriores ainda erram, com uma conta mais complicada para o resíduo; no `scikit-learn`, é o `GradientBoostingClassifier`, que volta na seção 13.6.

## Importância de Variáveis

> **📌 Nota**
>
> Esta seção corresponde à seção 8.2.1 de James et al. (2023).

Uma árvore de decisão se lê de cima para baixo: a raiz diz qual coluna separa melhor as observações, e cada nó abaixo diz a pergunta seguinte. Uma floresta de 300 árvores não se lê desse jeito, e a previsão dela é uma média que nenhum desenho resume. Dá para saber, ao menos, em quais colunas o modelo se apoia?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Quanto cada coluna reduz a impureza

Cada corte de uma árvore de classificação foi escolhido porque deixa os dois filhos mais puros que o nó, pelo índice de Gini da seção 12.3. A redução que o corte produz pesa pelo tamanho do nó: é o Gini do nó vezes o número de observações que chegam a ele, menos o mesmo produto nos dois filhos. Por causa desse peso, um corte na raiz, que separa todos os pacientes, conta mais que um corte num nó de poucos pacientes.

Somando essas reduções sobre os cortes que usam uma coluna, numa árvore, e depois tirando a média sobre as $B$ árvores, cada coluna ganha um número.

> **🔷 Conceito**
>
> A ***importância por redução de impureza*** de um preditor, num bagging ou numa floresta de $B$ árvores, é a redução total do índice de Gini (em classificação) ou do RSS (em regressão) nos cortes que usam esse preditor, somada dentro de cada árvore; a importância é a média dessas somas nas $B$ árvores. Um valor grande indica um preditor que o modelo usou muito para separar as observações de treino. Costuma-se mostrá-la relativa à maior, que vale 100.

Os pacientes do `Heart` são os da seção 13.2, com a mesma divisão: metade para o teste, sorteada com a semente 13 e com a mesma proporção de doentes nas duas metades. Desta vez as colunas de texto ficam como texto na tabela, e a troca por indicadoras vai para dentro do modelo:

In [ ]:
coracao = pd.read_csv("dados/Heart.csv").dropna()
y_c = coracao["doenca_cardiaca"]
X_cru = coracao.drop(columns="doenca_cardiaca")
treino_X, teste_X, treino_y, teste_y = train_test_split(
    X_cru, y_c, test_size=0.5, random_state=13, stratify=y_c)
print("treino:", len(treino_X), "  teste:", len(teste_X))
print("colunas:", X_cru.shape[1])

O modelo é um `Pipeline` de dois passos. O primeiro troca `dor_no_peito` e `talio` por uma indicadora para cada categoria e deixa as outras 11 das 13 colunas como estão; o segundo é o bagging de 300 árvores, isto é, a floresta com `max_features=None`:

In [ ]:
indicadoras = make_column_transformer(
    (OneHotEncoder(sparse_output=False), ["dor_no_peito", "talio"]),
    remainder="passthrough", verbose_feature_names_out=False,
).set_output(transform="pandas")
modelo = make_pipeline(
    indicadoras,
    RandomForestClassifier(n_estimators=300, max_features=None,
                           random_state=13))
modelo.fit(treino_X, treino_y)
print("categorias de dor_no_peito:",
      treino_X["dor_no_peito"].nunique())
print("colunas que as árvores veem:",
      len(modelo[0].get_feature_names_out()))

> **🔧 Função**
>
> **`make_column_transformer((transformador, colunas), remainder, verbose_feature_names_out)`** — aplica cada transformador só às colunas da lista dele e junta o resultado numa tabela. Com `remainder="passthrough"`, as outras colunas passam sem mudança; com `verbose_feature_names_out=False`, os nomes de saída não ganham prefixo. **`OneHotEncoder(sparse_output=False)`** — uma coluna de 0 e 1 para cada categoria; `sparse_output=False` devolve uma tabela comum, que é o que o `.set_output(transform="pandas")` precisa para montar um `DataFrame` com o nome de cada coluna.

> **🔧 Função**
>
> **`make_pipeline(passo1, passo2)`** — encadeia os passos: no `fit`, cada passo aprende com a saída do anterior; no `predict`, os dados passam pelos mesmos passos, já ajustados. **`modelo[0]`** e **`modelo[-1]`** — o primeiro e o último passo. **`transformador.get_feature_names_out()`** — os nomes das colunas que o passo ajustado produz. **`s.nunique()`** — o número de valores distintos.

As 13 colunas viraram 18: cada categoria de `dor_no_peito` e de `talio` virou uma indicadora, quatro para `dor_no_peito`. A floresta, `modelo[-1]`, guarda uma importância por coluna que viu, em `feature_importances_`. O chunk soma as indicadoras de volta na coluna de origem, para ter um número por coluna da tabela, e divide pela maior:

In [ ]:
por_indicadora = pd.Series(modelo[-1].feature_importances_,
                           index=modelo[0].get_feature_names_out())
origem = lambda nome: ("dor_no_peito" if nome.startswith("dor_no_peito")
                       else "talio" if nome.startswith("talio") else nome)
mdi = por_indicadora.groupby(origem).sum()
print("soma das importâncias = 1:", np.isclose(mdi.sum(), 1))
relativa = (100 * mdi / mdi.max()).sort_values(ascending=False)
print(relativa.map(lambda v: numero_br(v, 1)).to_string())

> **🔧 Função**
>
> **`modelo[-1].feature_importances_`** — a importância por redução de impureza de cada coluna que o modelo viu, na ordem das colunas, já como fração do total: as importâncias somam 1.
>
> **`s.groupby(funcao).sum()`** — aplica a função a cada rótulo da série, junta os rótulos que dão o mesmo resultado e soma os valores de cada grupo; aqui, as indicadoras de `dor_no_peito` somam na `dor_no_peito`, e as de `talio`, na `talio`.

A coluna de maior importância é `talio`, o exame com tálio, seguida de `depressao_st` e de `vasos_coloridos`; `colesterol` vem em quarto lugar, com 34,8 contra 100 de `talio`, e `glicemia_alta`, a última, fica em 1,2.

In [ ]:
# Figura: Importância por redução do índice de Gini de cada coluna do `Heart`, no bagging de 300 árvores ajustado nos 148 pacientes de treino, relativa à maior (100). As indicadoras de `dor_no_peito` e de `talio` estão somadas na coluna de origem.
fig, ax = plt.subplots(figsize=(4.4, 4.0))
ax.barh(range(len(relativa)), relativa.values, color="C0", height=0.7)
ax.set_yticks(range(len(relativa)), relativa.index, fontsize=8)
ax.set_ylim(len(relativa) - 0.5, -0.5)
ax.set_xlim(0, 105)
ax.set_xlabel("importância relativa à maior")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.set_ylim(baixo, cima)`** — os limites do eixo vertical; com o de baixo maior que o de cima, o eixo se inverte e a primeira barra, a de maior importância, fica no alto.

Essa importância é calculada com os pacientes de treino, os mesmos que as árvores usaram para escolher os cortes.

### Quanto o acerto cai se a coluna for embaralhada

`colesterol` aparece em quarto lugar. O modelo precisa dele para prever pacientes que não viu?

Uma forma de responder usa o teste. Embaralha-se uma coluna nos 149 pacientes de teste, deixando as outras como estão, e o modelo, já ajustado, prevê de novo. O embaralhamento quebra a ligação da coluna com a resposta e com as outras colunas e mantém os mesmos valores. Se o modelo depende da coluna, o acerto cai; se não depende, fica onde estava. A conta usa o teste, e não o treino, porque nos pacientes de treino as árvores já decoraram as respostas: a queda ali mediria o que o modelo memorizou, não o que serve para pacientes novos.

> **🔷 Conceito**
>
> A ***importância por permutação*** de um preditor é a queda na medida de desempenho do modelo ajustado, num conjunto de dados, quando os valores desse preditor são embaralhados entre as observações e os demais ficam como estão. O embaralhamento se repete várias vezes, e a importância é a média das quedas. Ela vale para qualquer modelo, porque só usa as previsões.

O `permutation_importance` faz isso com o `Pipeline` inteiro, e é aqui que ter as indicadoras dentro do modelo importa: ele embaralha a coluna `dor_no_peito` da tabela, e as quatro indicadoras dela mudam juntas, como mudariam num paciente de verdade. A medida é o acerto no teste, e cada coluna é embaralhada 20 vezes:

In [ ]:
perm = permutation_importance(modelo, teste_X, teste_y,
                              n_repeats=20, random_state=13)
queda = pd.DataFrame({"média": perm.importances_mean,
                      "desvio": perm.importances_std},
                     index=teste_X.columns)
queda = queda.loc[relativa.index]
print("acerto no teste:", numero_br(modelo.score(teste_X, teste_y), 4))
print(queda.map(lambda v: numero_br(v, 3)).to_string())

> **🔧 Função**
>
> **`permutation_importance(estimador, X, y, n_repeats, random_state)`** — embaralha cada coluna de `X`, uma por vez, `n_repeats` vezes, e mede a queda no `score` do estimador já ajustado; num classificador, o `score` é o acerto. Devolve um objeto com **`importances_mean`**, a média das quedas de cada coluna, **`importances_std`**, o desvio padrão delas entre as repetições, e **`importances`**, as quedas de cada repetição, uma linha por coluna. `random_state` fixa os embaralhamentos.
>
> **`df.loc[rotulos]`** — as linhas na ordem da lista `rotulos`; aqui, a ordem da importância por impureza.

O acerto no teste, 0,7584, pode diferir em um paciente do bagging da seção 13.2, com a mesma divisão: as mesmas 18 colunas chegam às árvores em outra ordem, e a ordem muda os desempates entre cortes igualmente bons.

A tabela está na ordem da importância por impureza. Um paciente a mais ou a menos de acerto, em 149, muda a taxa em 1/149, cerca de 0,007. Embaralhar `talio` derruba o acerto em 0,056, em média, perto de oito pacientes (0,056 × 149 ≈ 8,3). Em `depressao_st`, o desvio entre as 20 repetições, 0,029, é maior que a média, 0,015. Na ordem da permutação, as colunas ficam assim:

In [ ]:
por_permutacao = queda["média"].sort_values(ascending=False)
print("três primeiras:")
for coluna in por_permutacao.index[:3]:
    print(" ", coluna)
print("média negativa:")
for coluna in por_permutacao.index[por_permutacao < 0]:
    print(" ", coluna)
negativas = pd.Series((perm.importances < 0).sum(axis=1),
                      index=teste_X.columns)
print("repetições com queda negativa,")
for coluna in ["colesterol", "idade"]:
    print(f"  {coluna}: {negativas[coluna]} de 20")

> **🔧 Função**
>
> **`(m < 0).sum(axis=1)`** — em cada linha de `m`, quantos valores são negativos; aqui, em quantas das 20 repetições o acerto subiu.

As três primeiras pela permutação são `talio`, `vasos_coloridos` e `depressao_st`. Embaralhar `colesterol` não derrubou o acerto: a média das 20 quedas é −0,010, e em 15 das 20 repetições o modelo acertou mais com a coluna embaralhada do que com ela intacta. Com `idade`, 17 de 20. Uma média de −0,010 é cerca de um paciente e meio em 149 (0,010 × 149 ≈ 1,5). Uma queda negativa não quer dizer que a coluna atrapalha em geral. As 20 repetições mudam só o embaralhamento: o modelo e os 149 pacientes de teste são sempre os mesmos. O que elas mostram é que, neste modelo e nestes pacientes, trocar o colesterol de cada um pelo de outro não piora o acerto; em média o melhora um pouco, e na maior parte das repetições também: os cortes que as árvores fizeram em `colesterol` não ajudam a prever estes pacientes de teste. Com outro sorteio dos pacientes de teste, a média poderia sair diferente.

In [ ]:
# Figura: Queda no acerto do bagging nos 149 pacientes de teste do `Heart` quando cada coluna é embaralhada, média de 20 embaralhamentos, com um desvio padrão entre as repetições para cada lado (cinza). As colunas estão na ordem da importância por impureza da figura anterior. A linha vertical marca queda zero; à esquerda dela, o modelo acertou mais com a coluna embaralhada.
fig, ax = plt.subplots(figsize=(4.4, 4.0))
posicoes = range(len(queda))
ax.barh(posicoes, queda["média"], color="C1", height=0.7)
ax.errorbar(queda["média"], posicoes, xerr=queda["desvio"],
            fmt="none", ecolor="0.45", elinewidth=1, capsize=2)
ax.axvline(0, color="0.45", linewidth=0.8)
ax.set_yticks(posicoes, queda.index, fontsize=8)
ax.set_ylim(len(queda) - 0.5, -0.5)
ax.set_xlim(-0.03, 0.085)
ax.xaxis.set_major_locator(plt.MultipleLocator(0.02))
ax.set_xlabel("queda no acerto de teste")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.2f}".replace(".", ",")
                             .replace("-", "−"))
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.errorbar(x, y, xerr, fmt="none", ecolor, elinewidth, capsize)`** — um traço horizontal de `x - xerr` a `x + xerr` em cada `y`; com `fmt="none"`, sem marcador no centro. `capsize` é o tamanho da trava nas pontas. **`ax.axvline(x, color, linewidth)`** — uma linha vertical na posição `x`. **`ax.xaxis.set_major_locator(plt.MultipleLocator(0.02))`** — põe uma marca no eixo horizontal a cada 0,02. **`ax.xaxis.set_major_formatter(funcao)`** — escreve os números dessas marcas com a função.

Por que `colesterol` fica em quarto lugar numa medida e abaixo de zero na outra? A importância por impureza soma os cortes no treino, inclusive os fundos, perto das folhas, onde restam poucos pacientes. O chunk percorre as 300 árvores e guarda, para cada corte, a coluna, o número de pacientes de treino no nó e a redução ponderada do Gini. Depois conta os cortes de cada coluna, mede que porcentagem da importância dela vem de nós com até 20 pacientes (a coluna `%≤20`) e conta os valores distintos da coluna no treino (a coluna `dist.`):

In [ ]:
nomes = modelo[0].get_feature_names_out()
partes = []
for arvore in modelo[-1].estimators_:
    t = arvore.tree_
    peso = t.weighted_n_node_samples * t.impurity
    no = np.flatnonzero(t.feature >= 0)
    partes.append(pd.DataFrame({
        "coluna": [origem(nomes[k]) for k in t.feature[no]],
        "pacientes": t.n_node_samples[no],
        "reducao": peso[no] - peso[t.children_left[no]]
                   - peso[t.children_right[no]]}))
cortes = pd.concat(partes)
pequenos = cortes[cortes["pacientes"] <= 20]
resumo = pd.DataFrame({
    "cortes": cortes.groupby("coluna").size(),
    "%≤20": 100 * pequenos.groupby("coluna")["reducao"].sum()
            / cortes.groupby("coluna")["reducao"].sum(),
    "dist.": treino_X.nunique()})
linhas = list(relativa.index[:5]) + ["freq_cardiaca_max"]
print(resumo.loc[linhas].to_string(
    formatters={"%≤20": lambda v: numero_br(v, 1)}))
print("mais valores distintos:")
print(" ", resumo["dist."].idxmax())

> **🔧 Função**
>
> **`arvore.tree_`** — a árvore ajustada, em vetores com uma posição por nó; `tree_.feature >= 0` separa os nós que cortam das folhas. **`tree_.n_node_samples`** — as observações distintas da amostra bootstrap que chegam ao nó. **`tree_.weighted_n_node_samples`** — as mesmas, contadas com as repetições do sorteio. **`tree_.impurity`** — o Gini do nó.
>
> **`pd.concat(tabelas)`** — empilha as tabelas. **`df.groupby(coluna).size()`** — o número de linhas de cada grupo. **`df.to_string(formatters)`** — a tabela como texto, com uma função de formatação para as colunas do dicionário. **`s.idxmax()`** — o rótulo do maior valor.

`talio` tem 326 cortes, e só 2,1% da importância dele vem de nós com até 20 pacientes: ele separa grupos grandes. `colesterol` tem 775 cortes, e 62,7% da importância dele vem desses nós pequenos; em `idade`, 71,0%. Cada corte num nó pequeno reduz pouco o Gini, porque o peso é pequeno, mas somados eles fazem a maior parte da importância dessas duas colunas. `colesterol` tem ainda 100 valores distintos nos 148 pacientes de treino, o maior número entre as 13 colunas. São 99 pontos de corte possíveis, um entre cada par de valores vizinhos, para separar os últimos pacientes de um nó.

Valores distintos e cortes fundos não bastam para decidir o sinal da permutação. `freq_cardiaca_max` tem 69 valores distintos e 76,6% da importância vinda de nós pequenos, e embaralhá-la derruba o acerto em 0,006, em média, perto de um paciente (0,006 × 149 ≈ 0,9).

As duas medidas respondem a perguntas diferentes. A importância por impureza mede quanto o modelo usou cada coluna para separar os pacientes de treino, até os últimos de cada nó. A por permutação mede quanto o acerto em pacientes novos depende da coluna; com 149 pacientes, ela varia de um embaralhamento para outro, como mostra o desvio de `depressao_st`.

Nenhuma das duas diz que o colesterol não tem relação com doença cardíaca. Com as outras colunas à mão, este modelo não se apoia nele para acertar esses pacientes.

### Colunas que se substituem

As colunas de carreira do `Hitters` somam o que o jogador fez desde a estreia, e a seção 13.3 viu uma delas, `vezes_ao_bastao_carreira`, dominar as raízes do bagging. Duas colunas que carregam quase a mesma informação disputam os mesmos cortes. O que acontece com a importância de cada uma?

Os dados são os da seção 13.3: os jogadores com salário registrado, o logaritmo do salário como resposta e o treino de 132 jogadores sorteado com a semente 12.

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
y = np.log(hitters["salario"])
X = pd.get_dummies(hitters.drop(columns="salario"),
                   drop_first=True, dtype=float)
treino_Xh, teste_Xh, treino_yh, teste_yh = train_test_split(
    X, y, train_size=132, random_state=12)
carreira = [c for c in X.columns if c.endswith("_carreira")]
correlacao = X["vezes_ao_bastao_carreira"].corr(X["rebatidas_carreira"])
print("jogadores:", len(X), "  colunas:", X.shape[1])
print("treino:", len(treino_Xh), "  teste:", len(teste_Xh))
print("colunas de carreira:", len(carreira))
print("correlação entre vezes ao bastão")
print("  e rebatidas na carreira:", numero_br(correlacao, 3))
print("menor correlação entre as seis:",
      numero_br(X[carreira].corr().min().min(), 3))

> **🔧 Função**
>
> **`s.corr(outra)`** — a correlação entre duas séries. **`df.corr()`** — a tabela das correlações entre cada par de colunas; `.min().min()` é a menor delas.

Vezes ao bastão e rebatidas na carreira têm correlação de 0,995 nos 263 jogadores: saber uma é quase saber a outra. Entre as seis colunas de carreira, a menor correlação é 0,787. O chunk ajusta duas florestas de 300 árvores, o bagging ($m = p$) e a floresta com $m = 4$, e mede as duas importâncias para as seis colunas de carreira: a por impureza, relativa à maior entre as 19 colunas, e a por permutação nos 131 jogadores de teste, agora como aumento do MSE:

In [ ]:
importancias = {}
for nome, m in [("bagging", None), ("floresta", 4)]:
    ajuste = RandomForestRegressor(n_estimators=300, max_features=m,
                                   random_state=13)
    ajuste.fit(treino_Xh, treino_yh)
    imp = pd.Series(ajuste.feature_importances_, index=X.columns)
    perm_h = permutation_importance(
        ajuste, teste_Xh, teste_yh, n_repeats=10, random_state=13,
        scoring="neg_mean_squared_error")
    importancias[nome] = (100 * imp / imp.max(), pd.Series(
        perm_h.importances_mean, index=X.columns))
    print(f"{nome}, maior:")
    print(" ", imp.idxmax())
tabela = pd.DataFrame({nome: mdi_h[carreira].round()
                       for nome, (mdi_h, _) in importancias.items()})
tabela.index = [c.removesuffix("_carreira") for c in carreira]
print(tabela.astype(int).to_string())

In [ ]:
# Figura: As seis colunas de carreira do `Hitters` no bagging (azul) e na floresta com $m = 4$ (verde), 300 árvores cada, ajustados nos 132 jogadores de treino; nos dois painéis, a cor marca o modelo, não a medida. Em cima, a importância por impureza relativa à maior das 19 colunas; embaixo, o aumento do MSE de teste nos 131 jogadores de teste quando a coluna é embaralhada, média de 10 embaralhamentos.
nomes_c = [c.removesuffix("_carreira") for c in carreira]
posicoes = np.arange(len(carreira))
fig, eixos = plt.subplots(2, 1, figsize=(4.4, 5.6))
for eixo, qual, rotulo in [(eixos[0], 0, "importância relativa à maior"),
                           (eixos[1], 1, "aumento do MSE de teste")]:
    for desvio, (nome, cor) in zip([-0.2, 0.2], [("bagging", "C0"),
                                                 ("floresta", "C2")]):
        valores = importancias[nome][qual][carreira]
        eixo.barh(posicoes + desvio, valores, height=0.38, color=cor,
                  label=nome)
    eixo.set_yticks(posicoes, nomes_c, fontsize=8)
    eixo.set_ylim(len(carreira) - 0.5, -0.5)
    eixo.axvline(0, color="0.45", linewidth=0.8)
    eixo.set_xlabel(rotulo)
    eixo.xaxis.set_major_formatter(
        lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
eixos[0].set_xlim(0, 105)
eixos[1].xaxis.set_major_locator(plt.MultipleLocator(0.05))
eixos[1].legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`permutation_importance(..., scoring="neg_mean_squared_error")`** — mede a queda no MSE com sinal trocado, isto é, quanto o MSE de teste **aumenta** quando a coluna é embaralhada. **`s.astype(int)`** — a série convertida para inteiros; depois do `round()`, os valores arredondados sem o `.0`. Os nomes da tabela perdem o `_carreira` do fim, comum às seis colunas.

Nos dois modelos, a maior importância por impureza é a de `vezes_ao_bastao_carreira`. Os números da tabela podem mudar em uma unidade de um computador para outro. O chunk compara o que importa, com folga:

In [ ]:
mdi_bag, perm_bag = importancias["bagging"]
mdi_flo, perm_flo = importancias["floresta"]
print("rebatidas na carreira, impureza:")
print("  bagging abaixo de 35:", mdi_bag["rebatidas_carreira"] < 35)
print("  floresta acima de 85:", mdi_flo["rebatidas_carreira"] > 85)
print("bagging, aumento do MSE ao embaralhar")
print("  vezes ao bastão, mais de 0,05:",
      perm_bag["vezes_ao_bastao_carreira"] > 0.05)
print("  rebatidas, menos de 0,01:",
      perm_bag["rebatidas_carreira"] < 0.01)
outras = [c for c in carreira if c != "vezes_ao_bastao_carreira"]
print("floresta, as outras cinco de carreira")
print("  sobem na impureza:",
      (mdi_flo[outras] > mdi_bag[outras]).all())
print("floresta, aumento do MSE")
print("  vezes ao bastão < 0,05:",
      perm_flo["vezes_ao_bastao_carreira"] < 0.05)
print("  rebatidas > 0,01:",
      perm_flo["rebatidas_carreira"] > 0.01)

No bagging, em que toda divisão pode escolher qualquer coluna, `rebatidas_carreira` fica abaixo de 35 contra 100 de `vezes_ao_bastao_carreira`: com as duas entre as candidatas, um corte numa é quase tão bom quanto na outra, e a redução do corte vai inteira para a escolhida. Na floresta, `rebatidas_carreira` passa de 85, mas não sobe sozinha: as cinco colunas de carreira além de `vezes_ao_bastao_carreira`, que fica em 100 nos dois modelos, sobem todas. Com $m = 4$, a coluna mais forte fica fora do sorteio em muitas divisões, e qualquer outra ganha os cortes que sobram. Por isso a subida de `rebatidas_carreira` na impureza não mostra que ela substitui a outra; para ver isso, compara-se a permutação nos dois modelos.

No bagging, embaralhar `vezes_ao_bastao_carreira` aumenta o MSE de teste em mais de 0,05, e embaralhar `rebatidas_carreira`, em menos de 0,01. As árvores do bagging cortam por `vezes_ao_bastao_carreira`, e embaralhá-la desfaz os cortes em que elas se apoiam; `rebatidas_carreira` diz quase o mesmo, mas as árvores já ajustadas não cortam nela no lugar. Embaralhar `rebatidas_carreira` custa pouco pelo motivo inverso: as árvores a usam bem menos, e `vezes_ao_bastao_carreira`, intacta, carrega a informação. Na floresta, o aumento do MSE ao embaralhar `vezes_ao_bastao_carreira` fica abaixo de 0,05, e o de `rebatidas_carreira` passa de 0,01. Com $m = 4$, as árvores cortam em `rebatidas_carreira` nas divisões em que a outra não foi sorteada; embaralhar uma das duas desfaz só os cortes dela, e a outra segue carregando a informação nos cortes restantes. As duas dividem o trabalho.

Uma importância perto de zero, por qualquer das duas medidas, pode querer dizer que a coluna não ajuda a prever ou que outra coluna já diz o mesmo.

A importância de uma coluna é sempre a importância dela **dadas as outras**, num modelo ajustado com um treino. Ela não diz para que lado a coluna empurra a previsão: uma importância alta de `talio` não diz se um defeito no exame aumenta ou diminui a chance de doença. Nem diz que mudar o valor da coluna mudaria a resposta; mede quanto o modelo usa a coluna para prever.

## Resumo dos Métodos de Ensemble

> **📌 Nota**
>
> Esta seção corresponde à seção 8.2.5 de James et al. (2023).

Há três jeitos de juntar árvores: tirar a média de árvores ajustadas cada uma à sua amostra, tirar a média de árvores que, além disso, são obrigadas a diferir umas das outras, ou somar árvores pequenas, cada uma corrigindo o que as anteriores ainda erram. Nos mesmos jogadores e nos mesmos pacientes, com a mesma validação cruzada, quanto cada um erra, e quanto erram, ao lado deles, uma árvore só e um modelo linear?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import (GradientBoostingClassifier,
                              GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (GridSearchCV, KFold, StratifiedKFold,
                                     cross_val_score)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Três receitas

Os três métodos usam a árvore como ***aprendiz fraco*** (*weak learner*): um modelo que, sozinho, prevê mal ou oscila muito, e que o conjunto combina com outros. O toco da seção 13.4, com uma divisão só, prevê mal; a árvore sem poda da seção 13.1 muda muito de uma amostra de treino para outra. A árvore serve nesse papel porque acompanha limiares e interações sem que se escreva nenhum termo para eles. No método, preditores numéricos e categóricos entram na mesma árvore; no `scikit-learn`, os categóricos entram como indicadoras, feitas com `pd.get_dummies`.

O que muda de um método para outro é de onde vêm os dados de cada árvore, como as árvores se relacionam e o que sobra para escolher:

| método | dados | árvores | escolha |
|---|---|---|---|
| bagging | bootstrap | grandes | $B$ |
| floresta | bootstrap | grandes, $m$ colunas por divisão | $B$, $m$ |
| boosting | treino inteiro | pequenas, em sequência | $B$, $\lambda$, $d$ |

Nos três, $B$ é o número de árvores. Na floresta, $m$ é quantas colunas se sorteiam a cada divisão. No boosting, $\lambda$ é a taxa que encolhe cada árvore antes de ela entrar no modelo, e $d$ é o número de divisões de cada árvore: um toco tem $d = 1$. No bagging e na floresta, cada árvore é ajustada separadamente, na sua amostra bootstrap, sem olhar para as outras.

No bagging, as árvores só diferem porque as amostras diferem, e por isso se parecem: a seção 13.3 contou as raízes das árvores e achou quase sempre as mesmas colunas de carreira. A floresta sorteia as colunas candidatas a cada divisão, e árvores menos parecidas entram na média. O boosting não sorteia nada: cada árvore nova é ajustada aos resíduos que as anteriores deixaram e entra no modelo encolhida, de modo que o modelo aprende devagar.

### Nos jogadores e nos pacientes

A comparação usa os 263 jogadores de `Hitters` com salário registrado, com o logaritmo do salário como resposta, e os 297 pacientes de `Heart` com todas as medidas. Em cada conjunto, os cinco modelos passam pela mesma validação cruzada de 10 grupos.

Os parâmetros dos ensembles ficam fixos. A floresta dos jogadores sorteia $m = 4$ das 19 colunas, o $m$ que a seção 13.3 usou, perto de $\sqrt{19}$; a dos pacientes, o padrão `"sqrt"` do classificador. O boosting usa tocos e a taxa $\lambda = 0{,}01$ da seção 13.4, com um número de árvores dentro do trecho em que a curva de validação cruzada de lá ficou quase plana. Bagging e floresta têm 100 árvores cada. Uma árvore a mais só entra numa média, e por isso um $B$ maior não leva a sobreajuste (seção 13.4); o que ele custa é tempo de ajuste, multiplicado aqui pelos 10 grupos e pelos dois conjuntos.

In [ ]:
N_ARVORES = 100         # bagging e floresta
M_JOGADORES = 4         # colunas sorteadas por divisão
TAXA = 0.01             # λ do boosting
FOLHAS = 2              # tocos: d = 1
B_BOOSTING = 1000       # árvores do boosting

A árvore podada escolhe o `ccp_alpha` por validação cruzada de 5 grupos dentro de cada grupo externo, a validação cruzada aninhada da seção 12.4, com uma grade fixa que vai da árvore inteira à raiz sozinha. Os ensembles não escolhem nada: com os parâmetros fixos, basta um ajuste por grupo.

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
y = np.log(hitters["salario"])
X = pd.get_dummies(hitters.drop(columns="salario"),
                   drop_first=True, dtype=float)
grade = np.r_[0.0, np.logspace(-4, 0, 21)][::-1]
print("jogadores:", len(X), "  colunas:", X.shape[1])
print("grade:", len(grade), "valores, de",
      numero_br(grade[0], 0), "a", numero_br(grade[-1], 0))

> **🔧 Função**
>
> **`np.logspace(a, b, n)`** — `n` números igualmente espaçados em escala logarítmica, de $10^a$ a $10^b$; com o zero na frente e `[::-1]`, a grade vai de `ccp_alpha` = 1, a raiz sozinha, até 0, a árvore sem poda.

Antes de rodar: nos jogadores, a árvore podada errou menos que a regressão linear em todos os grupos da seção 12.4. Os ensembles, que tiram a média ou somam muitas árvores, devem ficar abaixo da árvore podada em todos os grupos, ou só na média?

In [ ]:
grupos = KFold(10, shuffle=True, random_state=13)
internos = KFold(5, shuffle=True, random_state=13)
modelos = {
    "árvore": GridSearchCV(DecisionTreeRegressor(random_state=13),
                           {"ccp_alpha": grade}, cv=internos,
                           scoring="neg_mean_squared_error"),
    "linear": LinearRegression(),
    "bagging": RandomForestRegressor(N_ARVORES, max_features=None,
                                     random_state=13),
    "floresta": RandomForestRegressor(N_ARVORES,
                                      max_features=M_JOGADORES,
                                      random_state=13),
    "boosting": GradientBoostingRegressor(
        n_estimators=B_BOOSTING, learning_rate=TAXA,
        max_leaf_nodes=FOLHAS, max_depth=None, random_state=13),
}
mse = pd.DataFrame({
    nome: -cross_val_score(modelo, X, y, cv=grupos,
                           scoring="neg_mean_squared_error")
    for nome, modelo in modelos.items()})
resumo_h = pd.DataFrame({"MSE": mse.mean(),
                         "desvio": mse.std() / np.sqrt(10)})
print(resumo_h.map(lambda v: numero_br(v, 4)).to_string())
print("abaixo da árvore, em quantos grupos:")
for nome in ["bagging", "floresta", "boosting"]:
    print(f"  {nome}: {(mse[nome] < mse['árvore']).sum()} de 10")

> **🔧 Função**
>
> **`LinearRegression()`** — a regressão linear por mínimos quadrados. **`df.std()`** — o desvio padrão de cada coluna, com $n - 1$ no denominador.

Com `scoring="neg_mean_squared_error"`, o `cross_val_score` devolve o MSE de cada grupo com o sinal trocado, e o `-` na frente o desfaz. A árvore entra nele como um `GridSearchCV`: cada ajuste num grupo externo é uma busca inteira, feita só com o treino daquele grupo.

O `desvio` é $s/\sqrt{10}$, com $s$ o desvio padrão dos MSE dos 10 grupos, como na seção 11.2. O $s$ mede quanto o MSE muda de um grupo para outro; o $s/\sqrt{10}$ dá uma ideia da incerteza da média dos 10. É uma régua grosseira, porque os 10 grupos não são medidas independentes: os treinos de dois grupos têm quase todas as linhas em comum.

In [ ]:
# Figura: Erro de validação cruzada de 10 grupos de cada modelo nos 263 jogadores do `Hitters`: MSE do logaritmo do salário. O ponto é a média dos 10 grupos, e o traço vai de um desvio $s/\sqrt{10}$ abaixo a um acima. Cores: árvore podada em cinza, regressão linear em roxo, bagging em azul, floresta aleatória em verde e boosting em laranja.
cores = {"árvore": "0.45", "linear": "C3", "logística": "C3",
         "bagging": "C0", "floresta": "C2", "boosting": "C1"}

def desenha(resumo, coluna, titulo, limite):
    fig, ax = plt.subplots(figsize=(3.8, 2.3))
    for posicao, nome in enumerate(resumo.index):
        # fmt="o": além do traço, um ponto no centro dele
        ax.errorbar(resumo.loc[nome, coluna], posicao,
                    xerr=resumo.loc[nome, "desvio"], fmt="o",
                    color=cores[nome], markersize=6, elinewidth=1.5,
                    capsize=3)
    ax.set_yticks(range(len(resumo)), resumo.index, fontsize=8)
    ax.set_ylim(len(resumo) - 0.5, -0.5)
    ax.set_xlim(0, limite)
    ax.set_title(titulo, loc="left", fontsize=9)
    ax.xaxis.set_major_formatter(
        lambda v, _: f"{v:.2f}".replace(".", ","))
    plt.tight_layout()
    plt.show()

desenha(resumo_h, "MSE", "Hitters, MSE", 0.55)

In [ ]:
mse_medio = resumo_h["MSE"]
print("menor MSE:", mse_medio.idxmin())
print("maior MSE:", mse_medio.idxmax())
print("árvore entre 0,28 e 0,34:",
      0.28 < mse_medio["árvore"] < 0.34)
ensembles = ["bagging", "floresta", "boosting"]
print("três ensembles abaixo de 0,22:",
      bool((mse_medio[ensembles] < 0.22).all()))
print("floresta abaixo do bagging:")
print("  na média:", mse_medio["floresta"] < mse_medio["bagging"])
print("  em mais de 6 dos 10 grupos:",
      (mse["floresta"] < mse["bagging"]).sum() > 6)
dif_h = mse["floresta"] - mse["bagging"]
print("distância entre as médias")
print("  abaixo de 0,03:", abs(dif_h.mean()) < 0.03)
print("desvio de cada um acima de 0,03:",
      bool((resumo_h.loc[["bagging", "floresta"], "desvio"] > 0.03).all()))
print("desvio da diferença")
print("  abaixo de 0,01:", dif_h.std() / np.sqrt(10) < 0.01)
print("  |média| acima de 2 desvios:",
      abs(dif_h.mean()) > 2 * dif_h.std() / np.sqrt(10))
baixo = mse_medio - resumo_h["desvio"]
cima = mse_medio + resumo_h["desvio"]
print("traços dos ensembles se cruzam:",
      baixo[ensembles].max() < cima[ensembles].min())
print("boosting menos outro,")
print("|média| abaixo de 1,5 desvio:")
for nome in ["bagging", "floresta"]:
    dif = mse["boosting"] - mse[nome]
    print(f"  {nome}:",
          abs(dif.mean()) < 1.5 * dif.std() / np.sqrt(10))

Nos jogadores, os três ensembles ficam abaixo de 0,22 de MSE, contra cerca de 0,3 da árvore podada e 0,4415 da regressão linear, que tem o maior MSE. Cada um deles erra menos que a árvore podada em cada um dos 10 grupos. Entre os três, a floresta tem o menor MSE médio e erra menos que o bagging em mais de 6 dos 10 grupos, o que repete, com outra divisão dos jogadores, o que a seção 13.3 mediu.

A segunda ou a terceira casa do MSE médio da árvore podada, e também a dos MSE médios do bagging e da floresta, não é firme. As árvores grandes dependem de desempates entre cortes igualmente bons, que uma diferença de arredondamento no último bit, de um computador para outro, decide para um lado ou para o outro, e a poda de cada grupo parte de uma árvore grande. O que vale comparar entre os modelos é a ordem e a contagem de grupos, que resistem a essas casas.

Os desvios do bagging e da floresta passam de 0,03 cada, e a distância entre as duas médias fica abaixo disso. Mas o $s$ de cada um, a variação do MSE de um grupo para outro, é em boa parte comum aos dois: calculada grupo a grupo, a diferença floresta − bagging tem desvio $s/\sqrt{10}$ abaixo de 0,01. Os grupos difíceis para um são difíceis também para o outro, e a diferença entre os dois oscila bem menos que o MSE de cada um. A comparação dentro de cada grupo diz então mais sobre a ordem entre dois modelos que as duas médias com os seus desvios. Contar grupos não basta para ordenar: o que separa a floresta do bagging é que a diferença média fica a mais de dois desvios de zero.

A comparação não é inteiramente simétrica. O $m = 4$ e o $B = 1.000$ foram lidos nas seções 13.3 e 13.4 com esses mesmos jogadores, o que favorece um pouco os ensembles, enquanto a árvore escolhe o `ccp_alpha` só com o treino de cada grupo. Contra a árvore, os três ganham nos 10 grupos, e na figura acima os três ficam à esquerda da árvore podada e da regressão linear. Entre eles, a comparação grupo a grupo sustenta a floresta à frente do bagging. O boosting fica perto dos dois: os traços dos três ensembles têm um trecho em comum, e, grupo a grupo, a diferença média entre o boosting e cada um deles fica abaixo de um desvio e meio. Com esta divisão dos jogadores em 10 grupos, não dá para pôr o boosting à frente nem atrás.

Nos pacientes, a disputa se repete com a árvore de classificação e a regressão logística sem penalização, como na seção 12.4: a árvore e os ensembles usam as indicadoras de todas as categorias, e a logística descarta a primeira de cada coluna de texto. O `cross_val_score` de um classificador devolve o acerto de cada grupo, e 1 menos o acerto é a taxa de erro.

A ordem dos jogadores se repete nos pacientes?

In [ ]:
coracao = pd.read_csv("dados/Heart.csv").dropna()
y_c = coracao["doenca_cardiaca"]
medidas = coracao.drop(columns="doenca_cardiaca")
X_arvores = pd.get_dummies(medidas, dtype=float)
X_logistica = pd.get_dummies(medidas, drop_first=True, dtype=float)
grupos = StratifiedKFold(10, shuffle=True, random_state=13)
internos = StratifiedKFold(5, shuffle=True, random_state=13)
modelos_c = {
    "árvore": GridSearchCV(DecisionTreeClassifier(random_state=13),
                           {"ccp_alpha": grade}, cv=internos,
                           scoring="accuracy"),
    "logística": LogisticRegression(C=np.inf, solver="newton-cholesky",
                                    tol=1e-8),
    "bagging": RandomForestClassifier(N_ARVORES, max_features=None,
                                      random_state=13),
    "floresta": RandomForestClassifier(N_ARVORES, max_features="sqrt",
                                       random_state=13),
    "boosting": GradientBoostingClassifier(
        n_estimators=B_BOOSTING, learning_rate=TAXA,
        max_leaf_nodes=FOLHAS, max_depth=None, random_state=13),
}
erro = pd.DataFrame({
    nome: 1 - cross_val_score(
        modelo, X_logistica if nome == "logística" else X_arvores,
        y_c, cv=grupos)
    for nome, modelo in modelos_c.items()})
resumo_c = pd.DataFrame({"erro": erro.mean(),
                         "desvio": erro.std() / np.sqrt(10)})
print("pacientes:", len(coracao))
print("colunas:", X_arvores.shape[1], "e", X_logistica.shape[1])
print(resumo_c.map(lambda v: numero_br(v, 4)).to_string())

> **🔧 Função**
>
> **`DecisionTreeClassifier(random_state)`** — a árvore de classificação. **`LogisticRegression(C, solver, tol)`** — a regressão logística; `C=np.inf` a ajusta sem penalização, e `solver="newton-cholesky"` com `tol=1e-8` leva ao mesmo resultado em qualquer máquina.
>
> **`GradientBoostingClassifier(...)`** — o boosting de árvores para classificação, com os mesmos parâmetros do regressor: `n_estimators` é $B$, `learning_rate` é $\lambda$ e `max_leaf_nodes` é $d + 1$. Com duas classes, cada árvore é ajustada, paciente a paciente, à diferença entre a classe observada, 0 ou 1, e a probabilidade que o modelo prevê até ali.

In [ ]:
# Figura: Taxa de erro de validação cruzada de 10 grupos de cada modelo nos 297 pacientes do `Heart`. O ponto é a média dos 10 grupos, e o traço vai de um desvio $s/\sqrt{10}$ abaixo a um acima. Cores como na figura dos jogadores, com a regressão logística em roxo. O eixo horizontal tem escala diferente da daquela figura.
desenha(resumo_c, "erro", "Heart, taxa de erro", 0.25)

In [ ]:
taxa_media = resumo_c["erro"]
print("menor taxa de erro:", taxa_media.idxmin())
print("maior taxa de erro:", taxa_media.idxmax())
diferenca = erro["boosting"] - erro["logística"]
print("boosting − logística:")
print("  média", numero_br(diferenca.mean(), 4))
print("  desvio", numero_br(diferenca.std() / np.sqrt(10), 4))
print("  boosting erra menos:",
      (diferenca < 0).sum(), "de 10")
print("  boosting erra mais:",
      (diferenca > 0).sum(), "de 10")
entre = taxa_media[["bagging", "floresta"]].between(
    taxa_media["boosting"], taxa_media["árvore"])
print("bagging e floresta entre o")
print("  boosting e a árvore:", bool(entre.all()))
baixo_c = taxa_media - resumo_c["desvio"]
cima_c = taxa_media + resumo_c["desvio"]
print("traços dos ensembles se cruzam:",
      baixo_c[ensembles].max() < cima_c[ensembles].min())
par = ["boosting", "logística"]
print("boosting e logística se cruzam:",
      baixo_c[par].max() < cima_c[par].min())

In [ ]:
abaixo = {nome: (erro[nome] < erro["árvore"]).sum()
          for nome in ensembles}
print("abaixo da árvore, em quantos grupos:")
for nome, k in abaixo.items():
    print(f"  {nome}: {k} de 10")
print("nenhum passa de 6:", max(abaixo.values()) <= 6)
print("empate boosting e logística:",
      (diferenca == 0).sum(), "de 10")

Nos pacientes, a ordem é outra. A logística tem a menor taxa de erro, 0,1582, e o boosting vem logo atrás, com 0,1682. A diferença entre os dois, 0,0100 em média, fica abaixo do desvio dessa média, 0,0149, calculado com as diferenças grupo a grupo: o boosting erra menos que a logística em 1 dos 10 grupos, mais em 4, e os dois erram o mesmo número de pacientes nos outros 5 (10 − 1 − 4 = 5). Com esses 297 pacientes, os dois ficam praticamente iguais. O bagging e a floresta ficam entre o boosting e a árvore podada, que tem a maior taxa de erro dos cinco, 0,2086. Aqui os ensembles não ficam abaixo da árvore em todos os grupos: nenhum passa de 6 dos 10.

Na figura acima, os traços dos três ensembles têm um trecho em comum, e também os do boosting e da logística. A logística, linear no logaritmo da chance, fica à esquerda de todos os modelos de árvores.

Nenhum método ficou em primeiro nos dois conjuntos. A seção 12.4 viu a árvore podada ganhar da regressão linear nos jogadores e perder da logística nos pacientes, e juntar árvores não mudou esse quadro: nos jogadores, a floresta tem o menor MSE médio; nos pacientes, a logística tem a menor taxa de erro, com o boosting a menos de um desvio dela. Num conjunto novo, qual método usar se decide nesse conjunto, com a validação cruzada, como nos jogadores e nos pacientes.

### O que se escolhe em cada um

Os parâmetros da comparação acima ficaram fixos, mas em cada método há o que escolher. Em qual dos três uma escolha ruim custa mais?

- **Bagging.** Só $B$, e basta ser grande: uma árvore a mais só entra numa média, e um $B$ grande não leva a sobreajuste. O erro OOB da seção 13.2 estima o erro de teste sem separar dados.
- **Floresta.** $B$, do mesmo jeito, e $m$. O erro OOB escolhe $m$ com um ajuste por valor, sem validação cruzada, como na seção 13.3. Os pontos de partida usuais são $m \approx \sqrt p$ em classificação e $m \approx p/3$ em regressão [Hastie et al. (2009), seção 15.3]. Nos jogadores, $\sqrt{19} \approx 4$ e $19/3 \approx 6$, e o menor erro OOB da seção 13.3 ficou num desses dois valores.
- **Boosting.** $B$, $\lambda$ e $d$, e os três interagem: $\lambda$ menor pede $B$ maior, e $B$ grande demais sobreajusta, como a seção 13.4 mediu. Não há OOB, porque toda árvore vê o treino inteiro; $B$ se escolhe por validação cruzada, e um ajuste com `staged_predict` por grupo dá a curva inteira.

Com muitos milhares de observações, o `GradientBoostingRegressor` fica lento, porque procura o corte de cada árvore entre todos os valores de cada coluna. O `scikit-learn` tem uma versão que agrupa os valores de cada coluna em faixas antes de procurar os cortes e é feita para esse caso:

```python
from sklearn.ensemble import HistGradientBoostingRegressor

modelo = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=500)
```

Os parâmetros têm outros nomes, `max_iter` no lugar de `n_estimators`, mas o que se escolhe é o mesmo: quantas árvores, com que taxa e de que tamanho.

No bagging, um $B$ pequeno demais custa o ruído de uma média com poucas árvores, e um $B$ grande demais, só tempo. Na floresta, $B$ é igual, e um $m$ ruim custa erro, mas o erro OOB mede cada $m$ com um ajuste só, sem separar dados. No boosting, $B$ custa erro dos dois lados e depende de $\lambda$ e de $d$, e nenhum atalho sem validação cruzada o escolhe: é o método em que a escolha pede mais cuidado. Floresta e boosting pedem uma estimativa do erro fora do treino: o OOB para $m$, a validação cruzada para $B$, $\lambda$ e $d$. Entre famílias de modelos, quem decide é a validação cruzada.

## Leituras adicionais

- O capítulo 8 de James et al. (2023). O PDF gratuito e os dados estão no [site oficial do livro](https://www.statlearning.com).
- A seção 8.7 de Hastie et al. (2009), sobre o bagging, e os capítulos 10, sobre o boosting e as árvores aditivas, e 15, sobre as florestas aleatórias, num tratamento mais técnico.
- Breiman (1994), o relatório técnico que propôs o bagging, com experimentos em árvores de classificação e de regressão.
- Breiman (2001)florestas, o artigo das florestas aleatórias, com o sorteio de colunas a cada divisão, o erro *out-of-bag* e a importância por permutação.
- Friedman (2001), que formula o boosting como uma descida de gradiente sobre funções, com a taxa de aprendizado e as árvores pequenas.
- Strobl et al. (2007), sobre o viés da importância por redução de impureza a favor de colunas com muitos valores distintos ou muitas categorias.
- O capítulo 7 de Géron (2022), "Ensemble Learning and Random Forests", com bagging, florestas e boosting no `scikit-learn`.
- [O guia de ensembles do scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html), com o `BaggingRegressor`, o `RandomForestRegressor`, o `GradientBoostingRegressor` e o `HistGradientBoostingRegressor`.
- [O guia de importância por permutação do scikit-learn](https://scikit-learn.org/stable/modules/permutation_importance.html) e [o exemplo que a compara com a importância por impureza](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance.html).

## Referências

- **Breiman**. *Bagging Predictors*. 1994.
- **Breiman**. *Statistical Modeling: The Two Cultures*. 2001.
- **Breiman**. *Random Forests*. 2001.
- **Friedman**. *Greedy Function Approximation: A Gradient Boosting Machine*. 2001.
- **Géron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed. O'Reilly Media. 2022.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed. Springer. 2009.
- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.
- **Strobl; Boulesteix; Zeileis; Hothorn**. *Bias in Random Forest Variable Importance Measures: Illustrations, Sources and a Solution*. 2007.